# OnKith Model V2 - DeBERTa-v3-xsmall

One linear, restartable run: build the data, train `microsoft/deberta-v3-xsmall`
on the canonical 63-label ontology, evaluate on six separate sets, export ONNX
and INT8, and produce the deployment package.

**Ontology.** 31 entity types, 63 BIO labels. Relative to the TinyBERT 67-label
head: `ALLERGIES`, `USERNAME` and `WEIGHT` are removed, `HEIGHT` stays out
(it was never in the head), and `PASSWORD` is added. `PASSWORD` is admitted only
for genuine conversational credential disclosure - see
`privacy_data_v2/password_rules.py`.

**Data.** AI4Privacy English is the base. Gretel, Nemotron, Kiji and the Saudi
generator are *targeted gap-fillers*: rows are selected because they strengthen
a weak label, never concatenated wholesale. Once a row is selected every
correctly mapped span in it is kept.

**Evaluation sets, in the order this notebook uses them.**

| # | Set | Purpose |
|---|---|---|
| 1 | training-side `test` | in-distribution generalisation: the historical 16,374 AI4Privacy rows **plus** the grouped supplementary test rows |
| 2 | `val_original` (40,909) | untouched AI4Privacy English validation |
| 3 | `val_supplementary` (20%) | weak-label validation, grouped holdout |
| 4 | combined validation | 2 + 3, the model-selection metric |
| 5 | OOD dev | error analysis, free to iterate on |
| 6 | frozen OOD final | **not run here**: a separate one-shot notebook, on the accepted INT8 candidate |

Only the AI4Privacy component of set 1 is comparable with the frozen TinyBERT
baseline. Section 20 restricts to exactly those 16,374 rows and to the 30 shared
entity types; the full Model V2 `test` number is reported separately beside it.

**Release-evaluation order.** This notebook builds a deployment candidate and
stops there. **Run All** is always safe: nothing here can touch the frozen OOD
final set.

```
train -> select best checkpoint -> FP32/PyTorch validation -> OOD dev
      -> save artifacts -> FP32 ONNX export -> FP32 parity -> INT8 quantization
      -> INT8 validation (test, val_original, val_supplementary,
         combined validation, OOD dev) -> INT8 quality gate
      -> accepted INT8 candidate manifest -> production-pin check
      -> deployment summary
```

The frozen benchmark lives in `model_v2_frozen_ood_final.ipynb`. It is spent
once, on the candidate this notebook accepts, and only after every number here
has been reviewed. Section 6 still fingerprints the frozen texts for the
contamination firewall; it never reads their annotations and never scores them.

**Restartability.** Every expensive stage writes an artifact and is skipped when
that artifact already matches the configuration. Set `FORCE_REBUILD_DATA = True`
to rebuild the corpora from scratch. A `latest.pt` that already recorded early
stopping is not resumed into another epoch; its best checkpoint is reused.

**What is committed.** Corpora, checkpoints, weights and ONNX graphs stay local
and gitignored. `config/`, `metrics/` and `tokenizer/` are the tracked evidence,
and `model_v2_data_manifest.json` is what lets a teammate regenerate an
identical supplementary holdout.

## 1. Environment

Verify the runtime before anything expensive happens. `PYTHONUTF8=1` matters on
Windows: without it the modern ONNX exporter dies while printing a progress
marker and the legacy fallback produces `If` subgraphs that cannot be quantized.

In [1]:
import os, platform, sys

os.environ.setdefault("PYTHONUTF8", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "0")   # set to "1" to forbid downloads

import gc, json, math, random, time, hashlib, tracemalloc
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "FullPipeline_V3").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "FullPipeline_V3").is_dir():
    raise RuntimeError("Run this notebook from inside the OnKith repository.")

for extra in (REPO_ROOT / "model" / "privacy_data_v2", REPO_ROOT / "model" / "privacy_evaluation"):
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

import transformers

print("repo root        :", REPO_ROOT)
print("python           :", platform.python_version(), "|", platform.platform())
print("torch            :", torch.__version__, "| cuda:", torch.cuda.is_available())
print("transformers     :", transformers.__version__)
try:
    import onnxruntime
    print("onnxruntime      :", onnxruntime.__version__)
except ImportError:
    print("onnxruntime      : not installed (needed from section 23 onwards)")
print("PYTHONUTF8       :", os.environ.get("PYTHONUTF8"))
if torch.cuda.is_available():
    print("gpu              :", torch.cuda.get_device_name(0))

c:\Users\yahya\VSCodeProjects\OnKith\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


repo root        : c:\Users\yahya\VSCodeProjects\OnKith
python           : 3.11.9 | Windows-10-10.0.26200-SP0
torch            : 2.11.0+cu128 | cuda: True
transformers     : 5.13.1
onnxruntime      : 1.27.0
PYTHONUTF8       : 1
gpu              : NVIDIA GeForce RTX 4050 Laptop GPU


## 2. Configuration

Everything tunable lives in this one cell. Seeds, paths, quotas, split factors,
training hyper-parameters and the INT8 quality gate.

In [2]:
SEED = 42

MODEL_DIR = REPO_ROOT / "model" / "DeBERTa-v3-xsmall"
FINAL_ARTIFACTS = MODEL_DIR / "artifacts" / "model_v2"
SMOKE_TEST = os.environ.get("MODEL_V2_SMOKE_TEST", "0") == "1"
ARTIFACTS = FINAL_ARTIFACTS / "smoke" if SMOKE_TEST else FINAL_ARTIFACTS
CONFIG_DIR = ARTIFACTS / "config" if SMOKE_TEST else MODEL_DIR / "config"
DATA_DIR      = ARTIFACTS / "data"          # gitignored
CHECKPOINTS   = ARTIFACTS / "checkpoints"   # gitignored
MODEL_STORE   = ARTIFACTS / "models"
HF_MODEL_DIR  = MODEL_STORE / "huggingface"  # gitignored
FP32_DIR      = MODEL_STORE / "fp32"         # gitignored; transfer or Git LFS
INT8_DIR      = MODEL_STORE / "int8"         # gitignored; release deployment binary
METRICS_DIR   = ARTIFACTS / "metrics"       # small JSON evidence
TOKENIZER_DIR = ARTIFACTS / "tokenizer"     # required deployment files
MODEL_CACHE_DIR = FINAL_ARTIFACTS / "cache"  # shared download cache, gitignored
for directory in (DATA_DIR, CHECKPOINTS, HF_MODEL_DIR, FP32_DIR, INT8_DIR, METRICS_DIR, TOKENIZER_DIR, CONFIG_DIR, MODEL_CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

TRAIN_PATH  = DATA_DIR / "model_v2_train.jsonl"
TEST_PATH   = DATA_DIR / "model_v2_test.jsonl"
VAL_SUPP_PATH = DATA_DIR / "model_v2_val_supplementary.jsonl"
VAL_ORIG_PATH = DATA_DIR / "model_v2_val_original.jsonl"
MANIFEST_PATH = CONFIG_DIR / "model_v2_data_manifest.json"
LABEL_MAP_PATH = CONFIG_DIR / "model_v2_label_map.json"
MAPPING_CSV_PATH = CONFIG_DIR / "model_v2_source_mapping.csv"
TRAINING_CONFIG_PATH = CONFIG_DIR / "model_v2_training_config.json"

# --- data build -------------------------------------------------------------
FORCE_REBUILD_DATA = SMOKE_TEST    # smoke runs never reuse production-sized data artifacts

# Corpora built by an older data pipeline must never be reused in silence. Data
# pipeline v3 added the cross-partition contamination filter and narrowed two
# split keys, so any holdout or train/test artifact from an earlier build no
# longer matches the code that would read it. Those are rebuilt automatically
# and the reason is printed; nothing has to be deleted by hand.
import manifest as manifest_module

CACHED_DATA_MANIFEST = (
    json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) if MANIFEST_PATH.exists() else None
)
DATA_CACHE_STALE = not manifest_module.cache_is_current(CACHED_DATA_MANIFEST)
REBUILD_SUPPLEMENTARY = FORCE_REBUILD_DATA or DATA_CACHE_STALE
STALE_CACHE_REASON = (
    manifest_module.describe_stale_cache(CACHED_DATA_MANIFEST, path=MANIFEST_PATH)
    if DATA_CACHE_STALE
    else ""
)
if DATA_CACHE_STALE:
    for stale_artifact in (TRAIN_PATH, TEST_PATH):
        if stale_artifact.exists():
            stale_artifact.unlink()

SAUDI_CSV = REPO_ROOT / "model" / "data" / "saudi_mena_pii_dataset.csv"
SAUDI_MANIFEST = REPO_ROOT / "model" / "data" / "saudi_mena_pii_manifest.json"
if not SAUDI_CSV.exists():
    raise FileNotFoundError(
        f"Required Saudi corpus is missing: {SAUDI_CSV}. "
        "Run model/data/saudi_PII_Builder.ipynb with Run All, then restart this notebook."
    )
HF_TOKEN = os.environ.get("HF_TOKEN") or None
HF_CACHE = os.environ.get("PII_CACHE_DIR") or str(REPO_ROOT / "model" / "data" / ".pii_hf_cache")
# Normal defaults remain production-sized. Smoke mode is opt-in through the environment.
SMOKE_SOURCE_ROWS = int(os.environ.get("MODEL_V2_SMOKE_SOURCE_ROWS", "512"))
MAX_SOURCE_ROWS = SMOKE_SOURCE_ROWS if SMOKE_TEST else None
SUPPLEMENTARY_SOURCES = ["gretel", "nemotron", "kiji", "saudi_mena"]

# --- model ------------------------------------------------------------------
BASE_MODEL = "microsoft/deberta-v3-xsmall"
BASE_MODEL_REVISION = "4b419818330868dff6a60ad3e6b1c730f8b8c0c6"
CONTEXT_LIMIT = 512      # production windowing contract
OVERFLOW_STRIDE = 128

# --- training ---------------------------------------------------------------
MAX_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 2
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
LEARNING_RATE = 3e-5     # below TinyBERT's 5e-5; deberta-v3 is a stronger backbone
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06      # deberta-v3 is warmup-sensitive
MAX_GRAD_NORM = 1.0
USE_AMP = torch.cuda.is_available()
AMP_INITIAL_SCALE = 4096.0
# Per-epoch validation is subsampled for speed; the selected checkpoint is
# always re-scored on the full sets in sections 17-21.
EPOCH_EVAL_SAMPLE = 6000
SMOKE_TRAIN_STEPS = int(os.environ.get("MODEL_V2_SMOKE_TRAIN_STEPS", "3"))
SMOKE_EVAL_ROWS = int(os.environ.get("MODEL_V2_SMOKE_EVAL_ROWS", "64"))
RESUME_TRAINING = not SMOKE_TEST

# --- tokenization memory ----------------------------------------------------
# The training split is windowed and materialised in memory. A bounded probe in
# section 12 projects the footprint before the full split is built.
TOKENIZATION_PROBE_ROWS = 64 if SMOKE_TEST else 2000
TOKENIZATION_MEMORY_WARN_GIB = float(os.environ.get("MODEL_V2_MEMORY_WARN_GIB", "16"))

# --- quantization gate ------------------------------------------------------
# INT8 is only deployable if it holds up on every view the model is judged on,
# out-of-distribution behaviour included.
INT8_MAX_TYPED_F1_DROP = 0.02
INT8_MAX_LEAKAGE_RISE = 0.015
INT8_GATE_VIEWS = ("test", "val_original", "val_supplementary",
                   "combined_validation", "ood_dev")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("mode             :", "SMOKE (isolated)" if SMOKE_TEST else "production")
print("artifacts        :", ARTIFACTS)
print("committed config:", CONFIG_DIR)
print("FORCE_REBUILD_DATA:", FORCE_REBUILD_DATA)
print("data pipeline    :", manifest_module.DATA_PIPELINE_VERSION,
      "| cached artifacts:", manifest_module.cache_pipeline_version(CACHED_DATA_MANIFEST) or "none")
print("frozen OOD final:", "never run here - see model_v2_frozen_ood_final.ipynb")
if STALE_CACHE_REASON:
    print("\n" + STALE_CACHE_REASON)

mode             : production
artifacts        : c:\Users\yahya\VSCodeProjects\OnKith\model\DeBERTa-v3-xsmall\artifacts\model_v2
committed config: c:\Users\yahya\VSCodeProjects\OnKith\model\DeBERTa-v3-xsmall\config
FORCE_REBUILD_DATA: False
data pipeline    : 3 | cached artifacts: 3
frozen OOD final: never run here - see model_v2_frozen_ood_final.ipynb


## 3. Canonical ontology

`privacy_data_v2/ontology.py` is the single source of truth and self-checks at
import. This cell writes the committed label map that the model config, the
ONNX export and every evaluator read back.

In [3]:
import ontology
from ontology import BIO_LABELS, ENTITIES, ID2LABEL, LABEL2ID, NUM_LABELS, SHARED_WITH_TINYBERT

assert len(ENTITIES) == 31 and NUM_LABELS == 63, (len(ENTITIES), NUM_LABELS)

payload = ontology.label_map_payload()
LABEL_MAP_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print(f"{len(ENTITIES)} entities / {NUM_LABELS} BIO labels")
print("removed :", ", ".join(sorted(ontology.REMOVED_ENTITIES)))
print("absent  :", ", ".join(sorted(ontology.ABSENT_ENTITIES)))
print("added   :", ", ".join(sorted(ontology.ADDED_ENTITIES)))
print("shared with TinyBERT:", len(SHARED_WITH_TINYBERT), "entities")
print("volume-weak labels  :", ", ".join(ontology.WEAK_LABELS))
print("OOD-weak labels     :", ", ".join(sorted(ontology.OOD_WEAK_LABELS)))
print("\nwrote", LABEL_MAP_PATH.relative_to(REPO_ROOT))

31 entities / 63 BIO labels
removed : ALLERGIES, USERNAME, WEIGHT
absent  : HEIGHT
added   : PASSWORD
shared with TinyBERT: 30 entities
volume-weak labels  : ACCOUNTNUM, AMOUNT, BANKNAME, COUNTRY, CURRENCY, IPV4, ORGANISATION, PASSWORD, SALARY, TIMEZONE, URL
OOD-weak labels     : DATE, DRIVERLICENSENUM, IDCARDNUM, ORGANISATION, TAXNUM, ZIPCODE

wrote model\DeBERTa-v3-xsmall\config\model_v2_label_map.json


## 4. Source registry and label mapping audit

Every source is declared with its licence and split names. Every native label
gets a decision with a written reason; nothing is ever guessed. This writes the
committed mapping CSV that a reviewer can read without running anything.

In [4]:
import label_mapping, sources
from label_mapping import mapping_decision

SAUDI_AUDIT = sources.audit_saudi_csv(SAUDI_CSV)
if SAUDI_MANIFEST.exists():
    saudi_manifest = json.loads(SAUDI_MANIFEST.read_text(encoding="utf-8"))
    recorded = saudi_manifest.get("artifacts", {}).get("dataset", {})
    assert recorded.get("sha256") == SAUDI_AUDIT["sha256"], "Saudi CSV hash does not match its manifest; rebuild it"
    assert recorded.get("rows") == SAUDI_AUDIT["rows"], "Saudi CSV row count does not match its manifest"
else:
    raise FileNotFoundError(
        f"Required Saudi manifest is missing: {SAUDI_MANIFEST}. Re-run model/data/saudi_PII_Builder.ipynb."
    )
print("Saudi corpus audit:", json.dumps(SAUDI_AUDIT, indent=2))

mapping_summary = label_mapping.write_mapping_csv(MAPPING_CSV_PATH)
print(json.dumps(mapping_summary, indent=2))

rows = label_mapping.mapping_table()
mapped = [row for row in rows if row["status"] == "mapped"]
targets = {row["onkith_label"] for row in mapped}
assert not targets - set(ENTITIES), f"mapping targets outside the ontology: {targets - set(ENTITIES)}"
print("\nontology labels with no source mapping:", sorted(set(ENTITIES) - targets) or "none")

print("\nmedium-confidence mappings needing review:")
for row in mapped:
    if row["confidence"] == "medium":
        gate = f"  [gate: {row['requires_span_gate']}]" if row["requires_span_gate"] else ""
        print(f"  {row['source_dataset']:12} {row['native_label']:28} -> {row['onkith_label']}{gate}")

print("\nrejected concepts (sample):")
for row in [r for r in rows if r["status"] != "mapped"][:12]:
    print(f"  {row['native_label']:30} {row['reason']}")

Saudi corpus audit: {
  "path": "C:\\Users\\yahya\\VSCodeProjects\\OnKith\\model\\data\\saudi_mena_pii_dataset.csv",
  "bytes": 21700096,
  "sha256": "811a28d99d3e7829bbf229712076a46359579fadd7f5e60bfa3fc8e1f158e54c",
  "rows": 40000,
  "spans": 109483,
  "columns": [
    "id",
    "text",
    "spans",
    "labels_present",
    "source_dataset",
    "source_split",
    "final_split",
    "is_saudi",
    "generation_type",
    "entity_source"
  ],
  "labels": {
    "ACCOUNTNUM": 7305,
    "BANKNAME": 6160,
    "BUILDINGNUM": 11177,
    "CITY": 15833,
    "COUNTRY": 3006,
    "CURRENCY": 4946,
    "GIVENNAME": 17050,
    "ORGANISATION": 9736,
    "PASSWORD": 2654,
    "STREET": 11717,
    "SURNAME": 78,
    "TAXNUM": 5976,
    "TELEPHONENUM": 5861,
    "TIMEZONE": 1326,
    "ZIPCODE": 6658
  },
  "password_rows": 2654,
  "password_spans": 2654,
  "rows_by_source_dataset": {
    "Saudi/MENA augmentation": 40000
  },
  "rows_by_generation_type": {
    "replacement": 14455,
    "synthetic":

## 5. AI4Privacy base dataset

The pinned English source, split exactly as TinyBERT split it, so the AI4Privacy
component of the training-side `test` set is the same 16,374 rows the frozen
baseline used.

* official English train **163,740** -> project train **147,366** + project
  validation **16,374** (the latter becomes the AI4Privacy component of Model V2
  `test`);
* official English validation **40,909** -> `val_original`, untouched.

The full Model V2 `test` set is those 16,374 rows **plus** the grouped
supplementary test rows added in section 9, so it is larger than - and not
interchangeable with - the TinyBERT comparison subset used in section 20.

`HEIGHT` (one span, validation only) and every other removed label are dropped
to `O`; the row itself is kept, so `val_original` keeps all 40,909 rows.

In [5]:
import data_audit
from records import make_record

def ai4privacy_records(dataset_split, indices, partition):
    """Convert a pinned AI4Privacy split into canonical Model V2 records."""
    supported = set(ENTITIES)
    out, dropped = [], Counter()
    if indices is None:
        subset = dataset_split
    elif hasattr(dataset_split, "select"):
        subset = dataset_split.select(indices)
    else:
        subset = [dataset_split[index] for index in indices]
    for row in subset:
        text = str(row["source_text"])
        spans = []
        for span in row["privacy_mask"]:
            label = str(span["label"])
            if label not in supported:
                dropped[label] += 1
                continue
            start, end = int(span["start"]), int(span["end"])
            if 0 <= start < end <= len(text) and text[start:end] == str(span["value"]):
                spans.append({"label": label, "start": start, "end": end, "value": text[start:end]})
            else:
                dropped[f"{label}:invalid_offset"] += 1
        spans.sort(key=lambda s: (s["start"], s["end"]))
        deduped, last_end = [], -1
        for span in spans:
            if span["start"] >= last_end:
                deduped.append(span); last_end = span["end"]
            else:
                dropped[f"{span['label']}:overlap"] += 1
        out.append(make_record(
            uid=f"ai4privacy:{row.get('uid', len(out))}",
            source_text=text, privacy_mask=deduped,
            source_dataset="ai4privacy", partition=partition,
            source_split=partition, region=str(row.get("region") or "") or None,
        ))
    return out, dropped


ai4_cache_paths = (VAL_ORIG_PATH, DATA_DIR / "ai4privacy_train.jsonl", DATA_DIR / "ai4privacy_test.jsonl")
if FORCE_REBUILD_DATA or not all(path.exists() for path in ai4_cache_paths):
    os.environ.setdefault("HF_HOME", HF_CACHE)
    if SMOKE_TEST:
        def sampled_english(split, target):
            rows = []
            for row in sources.iter_source_rows(
                "ai4privacy", split, hf_token=HF_TOKEN, cache_dir=HF_CACHE, limit=target * 32
            ):
                if str(row.get("language", "")).casefold() in {"en", "english"}:
                    rows.append(row)
                    if len(rows) >= target:
                        break
            if len(rows) < target:
                raise RuntimeError(f"Only found {len(rows)} English AI4Privacy rows in sampled {split}")
            return rows
        english_train = sampled_english("train", SMOKE_SOURCE_ROWS)
        english_validation = sampled_english("validation", max(32, SMOKE_SOURCE_ROWS // 4))
    else:
        from datasets import load_dataset
        dataset = load_dataset(sources.AI4PRIVACY["name"], revision=sources.AI4PRIVACY["revision"])
        english_train = dataset["train"].filter(lambda row: row["language"] == "en")
        english_validation = dataset["validation"].filter(lambda row: row["language"] == "en")
    print("official English train     :", len(english_train))
    print("official English validation:", len(english_validation))
    if not SMOKE_TEST:
        assert len(english_validation) == 40_909, "val_original must be the full 40,909 rows"

    privacy_masks = english_train["privacy_mask"] if hasattr(english_train, "column_names") else [row["privacy_mask"] for row in english_train]
    labels_by_row, _, indicator = data_audit.extract_row_entity_types(privacy_masks)
    if SMOKE_TEST:
        order = list(range(len(english_train))); random.Random(SEED).shuffle(order)
        cut = max(1, int(len(order) * 0.90)); train_idx, test_idx = order[:cut], order[cut:]
    else:
        train_idx, test_idx = data_audit.historical_multilabel_split(
            indicator, labels_by_row, validation_fraction=0.10, seed=42
        )
    print("project train / project validation:", len(train_idx), "/", len(test_idx))

    ai4_train, dropped_train = ai4privacy_records(english_train, train_idx, "train")
    ai4_test, _ = ai4privacy_records(english_train, test_idx, "test")
    val_original, dropped_val = ai4privacy_records(english_validation, None, "val_original")

    import records as records_module
    records_module.write_jsonl(VAL_ORIG_PATH, val_original)
    records_module.write_jsonl(DATA_DIR / "ai4privacy_train.jsonl", ai4_train)
    records_module.write_jsonl(DATA_DIR / "ai4privacy_test.jsonl", ai4_test)
    print("\nspans dropped to O (train):", dict(dropped_train.most_common(8)))
    print("spans dropped to O (val)  :", dict(dropped_val.most_common(8)))
else:
    import records as records_module
    ai4_train = records_module.load_jsonl(DATA_DIR / "ai4privacy_train.jsonl", validate=True)
    ai4_test = records_module.load_jsonl(DATA_DIR / "ai4privacy_test.jsonl", validate=True)
    val_original = records_module.load_jsonl(VAL_ORIG_PATH, validate=True)
    print("reused cached AI4Privacy artifacts")

print(f"\nai4_train={len(ai4_train):,}  ai4_test={len(ai4_test):,}  val_original={len(val_original):,}")
if not SMOKE_TEST:
    assert len(val_original) == 40_909

reused cached AI4Privacy artifacts

ai4_train=147,366  ai4_test=16,374  val_original=40,909


## 6. Contamination firewall

Built from every evaluation set **before** a single training row is written. Any
candidate whose normalised text already exists in an evaluation set is rejected
outright. The frozen OOD-final fixture is loaded here for fingerprints only -
its content is never read into training material and is never scored here.
The frozen benchmark is a separate one-shot notebook,
`model_v2_frozen_ood_final.ipynb`, run after an INT8 candidate is accepted.

In [6]:
import pools
from records import from_fixture_case
from privacy_eval.schema import load_cases

FIXTURES = REPO_ROOT / "model" / "privacy_evaluation" / "fixtures"
ood_dev = [from_fixture_case(case.to_dict(), partition="ood_dev")
           for case in load_cases(FIXTURES / "ood_dev.jsonl")]

# Verify the frozen file without constructing cases or exposing annotations. The
# firewall sees only streamed text and retains only normalized fingerprints.
OOD_FINAL_PATH = FIXTURES / "ood_final.jsonl"
OOD_FINAL_MANIFEST_PATH = FIXTURES / "ood_final.manifest.json"
ood_final_manifest = json.loads(OOD_FINAL_MANIFEST_PATH.read_text(encoding="utf-8"))
digest = hashlib.sha256(OOD_FINAL_PATH.read_bytes()).hexdigest()
assert digest == ood_final_manifest["fixture_sha256"], "Frozen OOD final fixture hash changed"
firewall = pools.build_firewall(val_original=val_original, ood_dev=ood_dev)
def frozen_final_texts_for_firewall():
    with OOD_FINAL_PATH.open("r", encoding="utf-8") as stream:
        for line in stream:
            if line.strip():
                yield str(json.loads(line)["text"])
final_case_count = sum(1 for line in OOD_FINAL_PATH.read_text(encoding="utf-8").splitlines() if line.strip())
assert final_case_count == ood_final_manifest["case_count"]
final_fingerprint_count = firewall.add_protected("ood_final", frozen_final_texts_for_firewall())

# Immutable identity of the frozen benchmark, recorded so the accepted-candidate
# manifest can tell the separate benchmark notebook which fixture this candidate
# was built against. Hashes and counts only - no case is ever constructed here.
FROZEN_FIXTURE_FINGERPRINT = {
    "path": OOD_FINAL_PATH.relative_to(REPO_ROOT).as_posix(),
    "manifest": OOD_FINAL_MANIFEST_PATH.relative_to(REPO_ROOT).as_posix(),
    "sha256": digest,
    "case_count": final_case_count,
    "generator_version": ood_final_manifest.get("generator_version"),
    "policy": ood_final_manifest.get("policy"),
}
print(f"firewall holds {firewall.size:,} protected fingerprints")
print(f"  val_original {len(val_original):,} | ood_dev {len(ood_dev)} | ood_final {final_case_count} cases/{final_fingerprint_count} newly protected")
print("\nood_final is fingerprinted only; this notebook never scores it.")
print("The frozen benchmark is model_v2_frozen_ood_final.ipynb, spent once on an")
print("accepted INT8 candidate.")

firewall holds 41,519 protected fingerprints
  val_original 40,909 | ood_dev 360 | ood_final 360 cases/318 newly protected

ood_final is fingerprinted only; this notebook never scores it.
The frozen benchmark is model_v2_frozen_ood_final.ipynb, spent once on an
accepted INT8 candidate.


## 7. Supplementary selection

Gretel, Nemotron, Kiji and the Saudi generator are read row by row. Each row is
mapped into the 63-label ontology, its `PASSWORD` spans are gated by
`password_rules`, and it is kept only if it strengthens a weak label - by volume
for labels the base cannot teach at all, by novelty for labels that have volume
but score badly out of distribution.

Selected rows keep **every** correctly mapped span, not just the triggering one.

If Nemotron is gated or its schema has drifted, it is skipped and the failure is
recorded; the build continues on the remaining sources.

In [7]:
import selection, dedup as dedup_module
from selection import SelectionConfig, SupplementarySelector

supp_training_side_path = DATA_DIR / "supplementary_training_side.jsonl"
supp_cache_paths = (VAL_SUPP_PATH, supp_training_side_path, MANIFEST_PATH)
if REBUILD_SUPPLEMENTARY or not all(path.exists() for path in supp_cache_paths):
    selector = SupplementarySelector(SelectionConfig())
    deduplicator = dedup_module.Deduplicator()
    mapping_audit = Counter()
    source_status = {}
    candidates = []

    for source in SUPPLEMENTARY_SOURCES:
        spec = sources.SUPPLEMENTARY_SOURCES[source]
        kept_here = 0
        try:
            if source == "saudi_mena":
                if not SAUDI_CSV.exists():
                    raise FileNotFoundError(
                        f"{SAUDI_CSV} is missing. Run model/data/saudi_PII_Builder.ipynb first."
                    )
                import itertools
                saudi_stream = sources.iter_saudi_rows(str(SAUDI_CSV))
                if MAX_SOURCE_ROWS is not None:
                    saudi_stream = itertools.islice(saudi_stream, MAX_SOURCE_ROWS)
                streams = [("generated", saudi_stream)]
            else:
                streams = [
                    (split, sources.iter_source_rows(
                        source, split, hf_token=HF_TOKEN,
                        cache_dir=HF_CACHE, limit=MAX_SOURCE_ROWS))
                    for split in spec["splits"]
                ]
            for split, stream in streams:
                for raw in stream:
                    record = sources.normalize_row(
                        source=source, row=raw, partition="candidate", audit=mapping_audit
                    )
                    if record is None:
                        continue
                    if not firewall.accept(record["source_text"]):
                        continue
                    kept, outcome = selector.consider(record)
                    if kept is None:
                        continue
                    keep, _ = deduplicator.check_and_add(kept["source_text"], kept["privacy_mask"])
                    if not keep:
                        continue
                    candidates.append(kept)
                    kept_here += 1
            source_status[source] = {"status": "ok", "selected_rows": kept_here,
                                     "revision": sources.resolve_revision(source, hf_token=HF_TOKEN)}
            if source == "saudi_mena":
                source_status[source]["artifact"] = {
                    **SAUDI_AUDIT,
                    "path": str(SAUDI_CSV.relative_to(REPO_ROOT)).replace(chr(92), "/"),
                }
        except Exception as exc:
            source_status[source] = {
                "status": "partial" if kept_here else "unavailable", "selected_rows": kept_here,
                "error": f"{type(exc).__name__}: {exc}",
            }
            print(f"  !! {source} {source_status[source]['status']}: {type(exc).__name__}: {exc}")
            if not spec.get("optional"):
                raise

    selection_audit = selector.audit()
    password_audit = __import__("password_rules").audit_password_decisions(selector.password_decisions)
    dedup_audit = deduplicator.audit()
    firewall_audit = firewall.audit()
else:
    candidates = None
    print("cached supplementary split found; section 8 will reuse it")

if candidates is not None:
    print(f"\nselected supplementary rows: {len(candidates):,}")
    print(json.dumps(source_status, indent=2))
    print("\nspans selected by label:")
    for label, count in sorted(selection_audit["selected_spans_by_label"].items()):
        carried = selection_audit["carried_spans_by_label"].get(label, 0)
        target = selection_audit["config"]["span_targets"].get(label)
        note = f" target={target}" if target else ""
        print(f"  {label:18} {count:7,}  (carried {carried:6,}){note}")
    if selection_audit["unmet_targets"]:
        print("\nunmet targets (measured evidence, not a failure to hide):")
        print(json.dumps(selection_audit["unmet_targets"], indent=2))
    print("\nPASSWORD gate:", json.dumps(password_audit, indent=2))
    print("\ndedup:", json.dumps(dedup_audit, indent=2))
    print("contamination firewall:", json.dumps(
        {k: v for k, v in firewall_audit.items() if k != "examples_by_protected_set"}, indent=2))

cached supplementary split found; section 8 will reuse it


## 8. Grouped 80/20 supplementary split

Rows are connected when they share **any** of the declared factors, so a
near-identical value or template cannot sit on both sides. The split is then
label-aware: every label with enough independent components contributes its
share to the holdout.

Two grouping keys are deliberately **narrow**, because a key that connects
unrelated rows protects nothing and starves the holdout:

* `email_domain` ignores public mail providers and the RFC 2606 reserved
  `example.*` names. Two rows both ending in `@gmail.com` share a provider, not
  a person; the whole address is still grouped by `open_entity_value`.
* `open_entity_value` ignores `DATE` and `BUILDINGNUM`. A calendar date and a
  house number are public facts, exactly like the closed vocabularies already
  excluded. Names are **not** excluded - a repeated `GIVENNAME` is the
  memorisation risk this split exists to control.

The cell reports which factor actually causes the largest component, so the
answer is measured rather than guessed.

The 20% holdout is written to a local, gitignored artifact and is regenerable
byte-for-byte from the committed manifest.

In [8]:
from grouped_splits import MODEL_V2_SPLIT_FACTORS
import records as records_module

if candidates is not None:
    supp_train, val_supplementary, supp_split_audit = pools.split_supplementary(
        candidates,
        holdout_fraction=pools.SUPPLEMENTARY_HOLDOUT_FRACTION,
        seed=pools.SEED_SUPPLEMENTARY_HOLDOUT,
        factors=MODEL_V2_SPLIT_FACTORS,
    )
    val_supp_artifact = records_module.write_jsonl(VAL_SUPP_PATH, val_supplementary)
    supp_training_artifact = records_module.write_jsonl(supp_training_side_path, supp_train)
else:
    if not MANIFEST_PATH.exists():
        raise FileNotFoundError(
            f"{MANIFEST_PATH} is missing, so the cached holdout cannot be verified. "
            "Set FORCE_REBUILD_DATA = True and re-run from section 7."
        )
    cached = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    manifest_module.assert_cache_current(cached, path=MANIFEST_PATH)
    supp_train = records_module.load_jsonl(supp_training_side_path, validate=True)
    val_supplementary = records_module.load_jsonl(VAL_SUPP_PATH, validate=True)
    supp_split_audit = cached["split"]["supplementary"]
    val_supp_artifact = cached["artifacts"]["val_supplementary"]
    supp_training_artifact = cached["artifacts"]["supplementary_training_side"]
    assert records_module.audit_jsonl(VAL_SUPP_PATH)["sha256"] == val_supp_artifact["sha256"], "cached supplementary holdout hash mismatch"
    assert records_module.audit_jsonl(supp_training_side_path)["sha256"] == supp_training_artifact["sha256"], "cached supplementary training-side hash mismatch"
    selection_audit = cached.get("selection", {})
    password_audit = cached.get("password", {})
    dedup_audit = cached.get("deduplication", {})
    source_status = cached.get("sources", {}).get("status", {})

print(f"supplementary training side: {len(supp_train):,}")
print(f"val_supplementary          : {len(val_supplementary):,}")
print(f"holdout sha256             : {val_supp_artifact['sha256']}")
print("\nfactors:", supp_split_audit["factors"])
print("disjoint on every factor:", supp_split_audit["is_disjoint"])
print("components:", supp_split_audit.get("component_count"),
      "| largest:", supp_split_audit.get("largest_component_rows"),
      f"({supp_split_audit.get('largest_component_fraction', 0):.2%} of the pool)")
if supp_split_audit.get("largest_component_fraction", 0) > 0.05:
    print("  NOTE: one component holds >5% of the pool. The diagnostics below name")
    print("  the cause; a component built from genuinely shared entity values is a")
    print("  property of the corpus, not a bug in the split.")
assert supp_split_audit["is_disjoint"], "a declared factor crossed the split boundary"

# The holdout exists only now, so this is the first moment it can join the
# firewall. From here the firewall covers every protected set, which is what
# makes the section 9 sweep complete rather than partial.
val_supp_fingerprints = firewall.add_protected_records("val_supplementary", val_supplementary)
print(f"\nfirewall now holds {firewall.size:,} protected fingerprints "
      f"(+{val_supp_fingerprints:,} from val_supplementary)")

diagnostics = supp_split_audit.get("component_diagnostics") or {}
if diagnostics:
    print("\nwhat connects the pool (largest component, as a share of the pool):")
    print(f"  {'factor':24} {'alone':>8} {'without it':>12}")
    for factor in supp_split_audit["factors"]:
        alone = diagnostics["per_factor"].get(factor, {})
        without = diagnostics["leave_one_out"].get(factor, {})
        print(f"  {factor:24} {alone.get('largest_component_fraction', 0):>8.2%}"
              f" {without.get('largest_component_fraction', 0):>12.2%}")
    print("\n  highest-degree connector keys:")
    for key, count in list(diagnostics["top_connector_keys"].items())[:8]:
        print(f"    {count:6,}  {key}")

thin = supp_split_audit.get("labels_without_independent_groups", [])
if thin:
    print("\nlabels without enough independent groups to validate this cycle:")
    print(" ", ", ".join(thin))
    print("  These are reported, not forced. Per-label claims for them are not supported.")

supplementary training side: 27,630
val_supplementary          : 6,907
holdout sha256             : 5ccac58b526738399c6e675083e9fd1bffbd51e052a2ce0ec999c529e1ccc79e

factors: ['open_entity_value', 'email_domain', 'canonical_template', 'organisation_family']
disjoint on every factor: True
components: 10578 | largest: 23203 (67.18% of the pool)
  NOTE: one component holds >5% of the pool. The diagnostics below name
  the cause; a component built from genuinely shared entity values is a
  property of the corpus, not a bug in the split.

firewall now holds 48,426 protected fingerprints (+6,907 from val_supplementary)

what connects the pool (largest component, as a share of the pool):
  factor                      alone   without it
  open_entity_value          61.29%        4.55%
  email_domain                0.08%       67.10%
  canonical_template          0.29%       67.18%
  organisation_family         4.44%       61.40%

  highest-degree connector keys:
       214  open_entity_value::

## 9. Contamination filter, training-side pool and train/test split

The firewall screens supplementary rows as they are *selected*, but the
AI4Privacy component is a **pinned** split rather than a selected one: whatever
upstream placed in the official train split arrives unscreened, duplicates
included. So before anything is split or written, every training-side pool is
swept against every protected set.

The rule when a training row and an evaluation row are the same text is fixed:
**the protected row is kept and the training-side copy is removed.**
`val_original` stays at its full 40,909 rows and the frozen OOD fixtures are
never touched.

The AI4Privacy component then keeps its pinned historical partition, so its test
rows stay comparable with the frozen TinyBERT baseline - minus any row the sweep
had to remove, which is reported explicitly rather than quietly retained to
preserve a round number.

In [9]:
# Sweep every training-side pool against every protected set before splitting.
# AI4Privacy is the pool that can carry an upstream duplicate, because its split
# is pinned rather than selected.
clean_pools, contamination_pre_split = pools.filter_contamination(
    firewall,
    {
        "ai4privacy_train": ai4_train,
        "ai4privacy_test": ai4_test,
        "supplementary_training_side": supp_train,
    },
    stage="pre_split",
)
ai4_train_clean = clean_pools["ai4privacy_train"]
ai4_test_clean = clean_pools["ai4privacy_test"]
supp_train_clean = clean_pools["supplementary_training_side"]

print(pools.render_contamination_audit(contamination_pre_split))
if contamination_pre_split["removals"]:
    print("\nwhat was removed (protected row kept, training copy dropped):")
    for removal in contamination_pre_split["removals"][:10]:
        print(f"  {removal['uid']} [{removal['candidate_partition']}]"
              f" -> {removal['protected_set']}:{removal['protected_uid']}"
              f"  chars={removal['text_length']} spans={removal['span_count']}")
    if contamination_pre_split["removals_truncated"]:
        print("  ... full list in the data manifest")

# The historical comparison subset, stated honestly. If the sweep had to remove
# rows from the pinned 16,374, the comparison set is smaller and section 20 says
# so; the number is not preserved by keeping contaminated rows.
AI4_TEST_HISTORICAL_ROWS = len(ai4_test)
AI4_TEST_COMPARISON_ROWS = len(ai4_test_clean)
AI4_TEST_ROWS_REMOVED = AI4_TEST_HISTORICAL_ROWS - AI4_TEST_COMPARISON_ROWS
print(f"\nAI4Privacy comparison subset: {AI4_TEST_COMPARISON_ROWS:,} of the historical "
      f"{AI4_TEST_HISTORICAL_ROWS:,} rows ({AI4_TEST_ROWS_REMOVED} removed as contaminated)")

train_rows, test_rows, training_split_audit = pools.split_training_side(
    ai4_train_clean, ai4_test_clean, supp_train_clean,
    test_fraction=pools.SUPPLEMENTARY_TEST_FRACTION,
    seed=pools.SEED_SUPPLEMENTARY_TEST,
    factors=MODEL_V2_SPLIT_FACTORS,
)
print("\n" + json.dumps({k: v for k, v in training_split_audit.items()
                  if k in {"ai4privacy_train_rows", "ai4privacy_test_rows",
                           "supplementary_train_rows", "supplementary_test_rows",
                           "total_train_rows", "total_test_rows", "is_disjoint"}}, indent=2))

firewall.assert_clean(train_rows, label="train")
firewall.assert_clean(test_rows, label="test")
print("\nfirewall: no train/test row appears in val_original, val_supplementary,")
print("ood_dev or ood_final. remaining protected overlap: 0")

cross-partition contamination audit
  stage: pre_split (filter v1, exact_normalized_text)
    ai4privacy_train             -> val_original: 7  val_supplementary: 0  ood_dev: 0  ood_final: 0
    ai4privacy_test              -> val_original: 1  val_supplementary: 0  ood_dev: 0  ood_final: 0
    supplementary_training_side  -> val_original: 0  val_supplementary: 0  ood_dev: 0  ood_final: 0
    removed from ai4privacy_train            7  (kept 147,359)
    removed from ai4privacy_test             1  (kept 16,373)
    removed from supplementary_training_side      0  (kept 27,630)
    removed at this stage             8

what was removed (protected row kept, training copy dropped):
  ai4privacy:24645970 [ai4privacy_train] -> val_original:ai4privacy:24794024  chars=0 spans=0
  ai4privacy:24645983 [ai4privacy_train] -> val_original:ai4privacy:24794024  chars=0 spans=0
  ai4privacy:24645985 [ai4privacy_train] -> val_original:ai4privacy:24794024  chars=0 spans=0
  ai4privacy:24645989 [ai4privacy

## 10. Training-only augmentation

One serious training run: real supplementary data **and** conservative,
span-safe augmentation together. Only field-proven transforms are used, each
rebuilding the text and recalculating offsets. No paraphrasing, no LLM
rewriting, no synonym substitution, no backtranslation - all of those can move
an entity boundary silently.

Two additions target measured failures:

* **PASSWORD disclosures**, generated with exact spans and re-checked through
  the same admission gate applied to external corpora;
* **hard negatives**, because AI4Privacy English contains only 44 no-PII rows in
  163,740 and the frozen baseline over-masks 65.28% of no-PII cases.

Caps keep this honest: at most 25% of the split augmented, at most 8% from any
one family, and every family must draw on at least 50 distinct parent rows.
Nothing here touches `test`, `val_original`, `val_supplementary` or either OOD
set - the guard in `augmentation._assert_mutable` rejects them outright.

In [10]:
import augment_pipeline
from augment_pipeline import AugmentationPlan, augment_training_rows

resources = augment_pipeline.load_resources()

# Entity values that appear in the protected holdout must not be reused as
# augmentation material, or the grouped split would leak through the back door.
holdout_values = defaultdict(set)
for record in val_supplementary + val_original:
    for span in record["privacy_mask"]:
        holdout_values[str(span["label"])].add(str(span["value"]))
resources = dict(resources)
resources["entity_values"] = augment_pipeline.restrict_inventories(
    resources["entity_values"], holdout_values
)
print("inventory sizes after holdout filtering:")
for label in ontology.WEAK_LABELS:
    print(f"  {label:18} {len(resources['entity_values'].get(label, []))}")

# --- generated additions ----------------------------------------------------
hard_negatives = augment_pipeline.generate_model_v2_hard_negatives(seed=SEED)
if SMOKE_TEST:
    hard_negatives = hard_negatives[:32]
target_hard_negatives = int(len(train_rows) * augment_pipeline.HARD_NEGATIVE_SHARE)
password_rows = augment_pipeline.generate_password_disclosures(
    resources["entity_values"], count=32 if SMOKE_TEST else 2000, seed=SEED
)
print(f"\ncurated hard-negative seeds : {len(hard_negatives)}")
print(f"hard-negative target rows   : {target_hard_negatives:,} (reached via span-safe variants below)")
print(f"password disclosure rows    : {len(password_rows):,}")

base_train = train_rows + hard_negatives + password_rows

plan = AugmentationPlan(base_rows=len(base_train), seed=SEED)
augmentation = augment_training_rows(base_train, resources, plan)
augmented_train = base_train + augmentation.rows
augmentation_audit = augmentation.audit(plan)

print("\n" + json.dumps({k: v for k, v in augmentation_audit.items() if k != "skipped_by_reason"}, indent=2))
if not augmentation_audit["parent_floor_satisfied"]:
    print("\nWARNING: these families drew on too few parents and are oversampling:")
    print(" ", ", ".join(augmentation_audit["families_below_parent_floor"]))

train_rows = augmented_train
print(f"\nfinal train rows: {len(train_rows):,}  (base {len(base_train):,} + augmented {len(augmentation.rows):,})")
empty = sum(1 for record in train_rows if not record["privacy_mask"])
print(f"no-PII rows in train: {empty:,} ({empty / len(train_rows):.2%})")

inventory sizes after holdout filtering:
  ACCOUNTNUM         4
  AMOUNT             6
  BANKNAME           2
  COUNTRY            4
  CURRENCY           11
  IPV4               8
  ORGANISATION       5
  PASSWORD           18
  SALARY             6
  TIMEZONE           8
  URL                6

curated hard-negative seeds : 146
hard-negative target rows   : 8,611 (reached via span-safe variants below)
password disclosure rows    : 1,762

{
  "plan": {
    "base_rows": 174134,
    "max_augmented_share": 0.25,
    "max_augmented_rows": 43533,
    "max_family_share": 0.08,
    "max_rows_per_family": 13930,
    "min_parents_per_family": 50,
    "families": [],
    "seed": 42
  },
  "augmented_rows": 43533,
  "rows_by_family": {
    "case_lower": 13930,
    "case_mixed": 1743,
    "case_upper": 13930,
    "entity_value": 13930
  },
  "distinct_parents_by_family": {
    "case_lower": 13919,
    "case_mixed": 1743,
    "case_upper": 13917,
    "entity_value": 13917
  },
  "families_below_par

## 11. Statistics, leakage proof and the reproducibility manifest

Augmentation ran after the section 9 sweep, so the final rows get one more pass
through the same filter before anything is written. Nothing is expected to be
removed here; the pass exists so that "the artifacts on disk are clean" is
proven rather than assumed.

The assertions here are the reason the generated corpora can stay out of Git:
they prove, rather than assume, that no evaluation row is a training row. The
manifest records everything a teammate needs to regenerate an identical
supplementary holdout, including the contamination audit and the data pipeline
version that invalidates older caches.

In [11]:
import manifest as manifest_module

# Final sweep, after augmentation, before the artifacts are written.
final_pools, contamination_pre_write = pools.filter_contamination(
    firewall, {"train": train_rows, "test": test_rows}, stage="pre_write",
)
train_rows, test_rows = final_pools["train"], final_pools["test"]
remaining_overlap = pools.assert_no_protected_overlap(
    firewall, {"train": train_rows, "test": test_rows}
)

contamination_audit = {
    "version": pools.CONTAMINATION_FILTER_VERSION,
    "stages": {
        "pre_split": contamination_pre_split,
        "pre_write": contamination_pre_write,
    },
    "removed_rows_total": (contamination_pre_split["removed_rows_total"]
                           + contamination_pre_write["removed_rows_total"]),
    "remaining_protected_overlap": remaining_overlap,
    "ai4privacy_comparison_subset": {
        "historical_rows": AI4_TEST_HISTORICAL_ROWS,
        "removed_as_contaminated": AI4_TEST_ROWS_REMOVED,
        "clean_rows": AI4_TEST_COMPARISON_ROWS,
        "note": (
            "The frozen TinyBERT aggregate was measured on all "
            f"{AI4_TEST_HISTORICAL_ROWS} historical rows. The Model V2 comparison "
            f"subset is the {AI4_TEST_COMPARISON_ROWS} of those that are not also "
            "in a protected evaluation set, so the two row sets are not identical "
            "and the comparison is close but not exact."
        ),
    },
}

print(pools.render_contamination_audit(contamination_pre_split, contamination_pre_write))
print(f"\nremoved from training side: {contamination_audit['removed_rows_total']:,}")
print(f"remaining protected overlap: {sum(remaining_overlap.values())}\n")

train_artifact = records_module.write_jsonl(TRAIN_PATH, train_rows)
test_artifact = records_module.write_jsonl(TEST_PATH, test_rows)

partitions = {
    "train": train_rows,
    "test": test_rows,
    "val_original": val_original,
    "val_supplementary": val_supplementary,
    "ood_dev": ood_dev,
}
disjointness = pools.assert_partitions_disjoint(partitions)
print("every partition is text-disjoint from every other partition")
print(json.dumps(disjointness["partitions"], indent=2))

statistics = {name: pools.partition_statistics(rows) for name, rows in partitions.items()}
print("\nspans by label (train vs the two validation sets):")
print(f"  {'label':18} {'train':>10} {'val_orig':>10} {'val_supp':>10}")
for label in ENTITIES:
    print(f"  {label:18} "
          f"{statistics['train']['spans_by_label'].get(label, 0):>10,} "
          f"{statistics['val_original']['spans_by_label'].get(label, 0):>10,} "
          f"{statistics['val_supplementary']['spans_by_label'].get(label, 0):>10,}")
print(f"\ntrain email domains: {statistics['train']['unique_email_domains']} "
      f"(the pinned AI4Privacy training distribution had only 7)")

def repo_relative_artifact(artifact):
    normalized = dict(artifact)
    normalized["path"] = str(Path(artifact["path"]).resolve().relative_to(REPO_ROOT)).replace(chr(92), "/")
    return normalized

artifacts_record = {
    "train": repo_relative_artifact(train_artifact),
    "test": repo_relative_artifact(test_artifact),
    "supplementary_training_side": repo_relative_artifact(supp_training_artifact),
    "val_supplementary": repo_relative_artifact(val_supp_artifact),
    "val_original": {"path": str(VAL_ORIG_PATH.relative_to(REPO_ROOT)).replace(chr(92), "/"), "rows": len(val_original),
                     "note": "Reconstructible from the pinned upstream revision; not required in Git."},
}

source_payload = sources.source_registry_payload(SUPPLEMENTARY_SOURCES)
source_payload["status"] = source_status

data_manifest = manifest_module.build_manifest(
    sources=source_payload,
    mapping_summary=mapping_summary,
    selection_audit=selection_audit,
    password_audit=password_audit,
    dedup_audit=dedup_audit,
    firewall_audit=firewall.audit(),
    supplementary_split_audit=supp_split_audit,
    training_split_audit=training_split_audit,
    augmentation_audit=augmentation_audit,
    partition_statistics=statistics,
    disjointness=disjointness,
    artifacts=artifacts_record,
    contamination_filter=contamination_audit,
    config={
        "seed": SEED, "base_model": BASE_MODEL, "base_model_revision": BASE_MODEL_REVISION,
        "context_limit": CONTEXT_LIMIT, "overflow_stride": OVERFLOW_STRIDE,
        "supplementary_sources": SUPPLEMENTARY_SOURCES,
        "max_source_rows": MAX_SOURCE_ROWS,
        "smoke_test": SMOKE_TEST,
        "frozen_ood_final": {"path": str(OOD_FINAL_PATH.relative_to(REPO_ROOT)).replace(chr(92), "/"),
                               "sha256": digest, "fingerprints": final_fingerprint_count,
                               "loaded_for_scoring": False},
        "split_factors": list(MODEL_V2_SPLIT_FACTORS),
        "supplementary_holdout_fraction": pools.SUPPLEMENTARY_HOLDOUT_FRACTION,
        "supplementary_test_fraction": pools.SUPPLEMENTARY_TEST_FRACTION,
        "contamination_filter_version": pools.CONTAMINATION_FILTER_VERSION,
    },
)
manifest_module.write_manifest(MANIFEST_PATH, data_manifest)
print("\nwrote", MANIFEST_PATH.relative_to(REPO_ROOT))
print("regenerate the holdout by re-running sections 1-11; verify with")
print("  privacy_data_v2.manifest.verify_manifest(committed, regenerated)")

cross-partition contamination audit
  stage: pre_split (filter v1, exact_normalized_text)
    ai4privacy_train             -> val_original: 7  val_supplementary: 0  ood_dev: 0  ood_final: 0
    ai4privacy_test              -> val_original: 1  val_supplementary: 0  ood_dev: 0  ood_final: 0
    supplementary_training_side  -> val_original: 0  val_supplementary: 0  ood_dev: 0  ood_final: 0
    removed from ai4privacy_train            7  (kept 147,359)
    removed from ai4privacy_test             1  (kept 16,373)
    removed from supplementary_training_side      0  (kept 27,630)
    removed at this stage             8
  stage: pre_write (filter v1, exact_normalized_text)
    train                        -> val_original: 0  val_supplementary: 0  ood_dev: 0  ood_final: 0
    test                         -> val_original: 0  val_supplementary: 0  ood_dev: 0  ood_final: 0
    removed from train                       0  (kept 217,667)
    removed from test                        0  (kept 19,136)

## 12. Tokenization and BIO alignment

The 512/128 windowing contract matches production exactly. Labels are aligned
from character spans onto token offsets, and **every token that covers source
characters is labelled** - sub-word continuations included. A continuation
inside an entity takes `I-<ENTITY>` (or `B-` when it opens the span), and a
token that straddles a span boundary is tagged with the entity rather than left
as `O`. `-100` is reserved for positions with no source characters: special
tokens here, and padding added later by the collator. Only those are ignored by
the loss.

Windowed features are materialised in memory, so a bounded probe measures the
per-row footprint first and projects it over the whole training split. The
offset mappings are released once section 12b has verified the alignment, which
frees roughly half of that footprint; `predict_spans` re-encodes on demand.

In [12]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL, revision=BASE_MODEL_REVISION, use_fast=True, cache_dir=MODEL_CACHE_DIR
)
assert tokenizer.is_fast, "a fast tokenizer is required for exact character offsets"


def encode_records(rows, *, desc="", progress_every=25_000):
    """Window and align one partition. Returns a list of feature dicts."""
    features = []
    started = time.perf_counter()
    for index, record in enumerate(rows):
        text = record["source_text"]
        encoded = tokenizer(
            text, truncation=True, max_length=CONTEXT_LIMIT, stride=OVERFLOW_STRIDE,
            return_overflowing_tokens=True, return_offsets_mapping=True,
            return_attention_mask=True, padding=False, return_token_type_ids=False,
        )
        spans = record["privacy_mask"]
        for window in range(len(encoded["input_ids"])):
            offsets = encoded["offset_mapping"][window]
            labels, previous_entity = [], None
            for start, end in offsets:
                if start == end:
                    labels.append(-100); previous_entity = None; continue
                tag = "O"
                for span in spans:
                    if start >= int(span["start"]) and end <= int(span["end"]):
                        prefix = "B" if start == int(span["start"]) else "I"
                        tag = f"{prefix}-{span['label']}"
                        break
                    if start < int(span["end"]) and end > int(span["start"]):
                        # Partial overlap: the token straddles a boundary. Tag it
                        # with the entity so the character is never left as O.
                        prefix = "B" if start <= int(span["start"]) else "I"
                        tag = f"{prefix}-{span['label']}"
                        break
                if tag.startswith("I-") and previous_entity != tag[2:]:
                    tag = "B-" + tag[2:]
                labels.append(LABEL2ID[tag])
                previous_entity = None if tag == "O" else tag[2:]
            features.append({
                "input_ids": encoded["input_ids"][window],
                "attention_mask": encoded["attention_mask"][window],
                "labels": labels,
                "offset_mapping": offsets,
                "row_index": index,
            })
        if desc and progress_every and index and index % progress_every == 0:
            print(f"  {desc}: {index:,}/{len(rows):,} rows | {len(features):,} windows | "
                  f"{time.perf_counter() - started:.0f}s")
    if desc:
        print(f"  {desc}: {len(rows):,}/{len(rows):,} rows | {len(features):,} windows | "
              f"{time.perf_counter() - started:.0f}s")
    return features


def release_offset_mappings(features):
    """Drop per-window offsets once alignment is verified.

    Offsets are the largest part of a materialised feature and are only needed by
    the section 12b checks; `predict_spans` re-encodes the rows it scores.
    """
    tuple_bytes = sys.getsizeof((0, 0))
    freed = 0
    for feature in features:
        offsets = feature.pop("offset_mapping", None)
        if offsets is not None:
            freed += sys.getsizeof(offsets) + len(offsets) * tuple_bytes
    gc.collect()
    return freed


# --- bounded memory probe before the full split is materialised -------------
probe_rows = train_rows[:min(TOKENIZATION_PROBE_ROWS, len(train_rows))]
gc.collect()
tracemalloc.start()
probe_features = encode_records(probe_rows)
probe_bytes = tracemalloc.get_traced_memory()[0]
tracemalloc.stop()
probe_windows = len(probe_features)
del probe_features
gc.collect()

bytes_per_row = probe_bytes / max(1, len(probe_rows))
projected_gib = bytes_per_row * len(train_rows) / 1024 ** 3
print(f"memory probe: {len(probe_rows):,} rows -> {probe_windows:,} windows, "
      f"{probe_bytes / 1024 ** 2:.1f} MiB retained ({bytes_per_row / 1024:.1f} KiB/row)")
print(f"projected train features for {len(train_rows):,} rows: about {projected_gib:.2f} GiB "
      f"(section 12b then releases the offsets, roughly halving it)")
available_gib = None
try:
    import psutil
    available_gib = psutil.virtual_memory().available / 1024 ** 3
    print(f"system memory available: {available_gib:.2f} GiB")
except ImportError:
    print("psutil not installed; system memory headroom not measured")
if projected_gib > TOKENIZATION_MEMORY_WARN_GIB or (
        available_gib is not None and projected_gib > 0.5 * available_gib):
    print("  WARNING: the projected footprint is large for this machine. Close other")
    print("  applications, or rebuild section 10 with fewer augmented rows, before")
    print("  starting a full run - tokenization happens before the first epoch.")

train_features = encode_records(train_rows, desc="train")
print(f"train windows: {len(train_features):,} from {len(train_rows):,} rows")
lengths = np.array([len(f["input_ids"]) for f in train_features])
print("token length p50/p90/p95/p99/max:",
      *[int(np.percentile(lengths, p)) for p in (50, 90, 95, 99)], int(lengths.max()))

memory probe: 2,000 rows -> 2,000 windows, 23.6 MiB retained (12.1 KiB/row)
projected train features for 217,667 rows: about 2.51 GiB (section 12b then releases the offsets, roughly halving it)
system memory available: 4.18 GiB
  applications, or rebuild section 10 with fewer augmented rows, before
  starting a full run - tokenization happens before the first epoch.
  train: 25,000/217,667 rows | 25,005 windows | 25s
  train: 50,000/217,667 rows | 50,015 windows | 50s
  train: 75,000/217,667 rows | 75,019 windows | 76s
  train: 100,000/217,667 rows | 100,033 windows | 103s
  train: 125,000/217,667 rows | 125,047 windows | 128s
  train: 150,000/217,667 rows | 150,052 windows | 152s
  train: 175,000/217,667 rows | 176,504 windows | 201s
  train: 200,000/217,667 rows | 201,722 windows | 235s
  train: 217,667/217,667 rows | 219,563 windows | 257s
train windows: 219,563 from 217,667 rows
token length p50/p90/p95/p99/max: 77 201 258 488 512


### 12b. Alignment sanity checks

These must pass before training. A silent alignment bug produces a model that
looks fine on token accuracy and fails completely on spans.

In [13]:
if not train_features or "offset_mapping" not in train_features[0]:
    raise RuntimeError(
        "train_features no longer carries offset mappings: they are released at the "
        "end of this section to save memory. Re-run section 12 before re-running "
        "these checks."
    )


def check_alignment(rows, features, sample=400):
    """Every gold span must be covered by at least one window, with the right tag."""
    by_row = defaultdict(list)
    for feature in features:
        by_row[feature["row_index"]].append(feature)

    checked = uncovered = mismatched = 0
    rng = random.Random(SEED)
    indices = rng.sample(range(len(rows)), min(sample, len(rows)))
    for index in indices:
        record = rows[index]
        covered_characters = set()
        for feature in by_row[index]:
            for (start, end), label_id in zip(feature["offset_mapping"], feature["labels"]):
                if label_id in (-100, LABEL2ID["O"]) or start == end:
                    continue
                label = ID2LABEL[label_id]
                entity = label[2:]
                matching = [s for s in record["privacy_mask"]
                            if str(s["label"]) == entity
                            and start < int(s["end"]) and end > int(s["start"])]
                if not matching:
                    mismatched += 1
                else:
                    covered_characters.update(range(start, end))
        for span in record["privacy_mask"]:
            checked += 1
            wanted = {position for position in range(int(span["start"]), int(span["end"]))
                      if not record["source_text"][position].isspace()}
            if not wanted or not wanted.issubset(covered_characters):
                uncovered += 1
    return {"spans_checked": checked, "uncovered_spans": uncovered, "mismatched_tokens": mismatched}


alignment = check_alignment(train_rows, train_features)
print(json.dumps(alignment, indent=2))
assert alignment["uncovered_spans"] == 0, "a gold span was not covered by any window"
assert alignment["mismatched_tokens"] == 0, "a token was tagged with the wrong entity"

# No orphan I- tags, and every label id is inside the 63-label range.
for feature in train_features if SMOKE_TEST else train_features[:5000]:
    previous = None
    for label_id in feature["labels"]:
        assert label_id == -100 or 0 <= label_id < NUM_LABELS
        if label_id == -100:
            previous = None
            continue
        label = ID2LABEL[label_id]
        if label.startswith("I-"):
            assert previous is not None and previous[2:] == label[2:], f"orphan {label}"
        previous = None if label == "O" else label
print("no orphan I- tags; all label ids inside [0, 62]")

example = train_features[0]
print("\nsample window (sub-word continuations carry I- tags; -100 is IGN):")
for token, (start, end), label_id in list(zip(
        tokenizer.convert_ids_to_tokens(example["input_ids"]),
        example["offset_mapping"], example["labels"]))[:24]:
    tag = "IGN" if label_id == -100 else ID2LABEL[label_id]
    print(f"  {token:<18} {str((start, end)):<12} {tag}")

# Alignment is proven, so the offsets can go. Training only needs ids/mask/labels.
freed_bytes = release_offset_mappings(train_features)
print(f"\nreleased training offset mappings: about {freed_bytes / 1024 ** 3:.2f} GiB freed")

{
  "spans_checked": 2855,
  "uncovered_spans": 0,
  "mismatched_tokens": 0
}
no orphan I- tags; all label ids inside [0, 62]

sample window (sub-word continuations carry I- tags; -100 is IGN):
  [CLS]              (0, 0)       IGN
  ▁Before            (0, 6)       O
  ▁the               (6, 10)      O
  ▁quarterly         (10, 20)     O
  ▁review            (20, 27)     O
  ,                  (27, 28)     O
  ▁Mrs               (28, 32)     B-TITLE
  ▁Sev               (32, 36)     B-GIVENNAME
  ina                (36, 39)     I-GIVENNAME
  ▁Tri               (39, 43)     I-GIVENNAME
  nge                (43, 46)     I-GIVENNAME
  ▁Sam               (46, 50)     B-SURNAME
  itsch              (50, 55)     I-SURNAME
  ▁Ghaz              (55, 60)     I-SURNAME
  r                  (60, 61)     I-SURNAME
  aoui               (61, 65)     I-SURNAME
  ▁sends             (65, 71)     O
  ▁a                 (71, 73)     O
  ▁reminder          (73, 82)     O
  ▁to                (82, 85)     

## 13. Load DeBERTa-v3-xsmall with the 63-label head

The preflight established the deployment contract at 67 labels with a random
head. Only the head width changes here (25,795 -> 24,255 parameters, about
0.002% of the model), so every tokenizer, ONNX, quantization and runtime result
from the preflight still applies.

In [14]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL, revision=BASE_MODEL_REVISION,
    cache_dir=MODEL_CACHE_DIR,
    num_labels=NUM_LABELS,
    id2label={index: label for index, label in ID2LABEL.items()},
    label2id=dict(LABEL2ID),
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# The upstream checkpoint is stored in FP16. Keep master weights and gradients
# in FP32; autocast handles mixed-precision activations safely on CUDA.
model.to(device=device, dtype=torch.float32)

total = sum(p.numel() for p in model.parameters())
head = sum(p.numel() for p in model.classifier.parameters())
print(f"{BASE_MODEL} @ {BASE_MODEL_REVISION[:12]}")
print(f"parameters: {total:,} (backbone {total - head:,} + head {head:,})")
print(f"labels    : {model.config.num_labels}")
print(f"device    : {device}")
assert model.config.num_labels == 63

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 9901.66it/s]
[transformers] DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
classifier.b

microsoft/deberta-v3-xsmall @ 4b4198183308
parameters: 70,706,367 (backbone 70,682,112 + head 24,255)
labels    : 63
device    : cuda


## 14. Batching

Dynamic padding to the longest window in each batch, rounded to a multiple of 8.
Labels pad with `-100`. The training-side percentiles from section 12 show p50
around 75 tokens, so padding to the batch maximum costs far less than padding to
512.

In [15]:
from torch.utils.data import DataLoader, Dataset


class WindowDataset(Dataset):
    def __init__(self, features):
        self.features = features

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        feature = self.features[index]
        return {
            "input_ids": feature["input_ids"],
            "attention_mask": feature["attention_mask"],
            "labels": feature["labels"],
        }


def collate(batch):
    longest = max(len(item["input_ids"]) for item in batch)
    longest = int(math.ceil(longest / 8) * 8)
    pad_id = tokenizer.pad_token_id
    out = {"input_ids": [], "attention_mask": [], "labels": []}
    for item in batch:
        padding = longest - len(item["input_ids"])
        out["input_ids"].append(item["input_ids"] + [pad_id] * padding)
        out["attention_mask"].append(item["attention_mask"] + [0] * padding)
        out["labels"].append(item["labels"] + [-100] * padding)
    return {key: torch.tensor(value, dtype=torch.long) for key, value in out.items()}


generator = torch.Generator(); generator.manual_seed(SEED)
train_loader = DataLoader(
    WindowDataset(train_features), batch_size=TRAIN_BATCH_SIZE, shuffle=True,
    collate_fn=collate, generator=generator, num_workers=0,
)
print(f"train batches per epoch: {len(train_loader):,}")

train batches per epoch: 13,723


## 15. Span prediction

Decoding reuses the **production** functions - `consolidate_predictions_to_words`
for whole-word promotion and the same BIO repair - so an evaluation number here
reflects what the deployed pipeline would actually emit, not a notebook-only
decoding path. The structured regex layer is deliberately excluded: these
numbers measure what the neural model learned.

In [34]:
sys.path.insert(0, str(REPO_ROOT / "FullPipeline_V3"))

# The span decoder and the ONNX masker are imported from FullPipeline_V3 and
# model/privacy_evaluation; nothing in this notebook defines them. A kernel that
# imported those packages before the span-boundary fixes still holds the old
# functions, so drop both from the module cache and let the imports below - and
# sections 17-27 - bind against the current source. Purging the whole package
# rather than reloading one module keeps every module consistent with its
# siblings. A fresh kernel is unaffected.
for _stale_module in [name for name in list(sys.modules)
                      if name.split(".")[0] in ("integrated_pipeline", "privacy_eval")]:
    del sys.modules[_stale_module]

from integrated_pipeline.span_merger import consolidate_predictions_to_words
from privacy_eval.model_v2_adapter import repair_bio


@torch.no_grad()
def predict_spans(rows, *, batch_size=EVAL_BATCH_SIZE, torch_model=None):
    """Predict character spans for canonical records, production-style."""
    active = torch_model if torch_model is not None else model
    active.eval()
    features = encode_records(rows)
    by_row = defaultdict(list)
    for feature in features:
        by_row[feature["row_index"]].append(feature)

    logits_by_feature = {}
    order = sorted(range(len(features)), key=lambda i: len(features[i]["input_ids"]))
    for start in range(0, len(order), batch_size):
        chunk = [features[i] for i in order[start:start + batch_size]]
        batch = collate([
            {"input_ids": f["input_ids"], "attention_mask": f["attention_mask"], "labels": f["labels"]}
            for f in chunk
        ])
        batch = {key: value.to(device) for key, value in batch.items() if key != "labels"}
        with torch.autocast("cuda", enabled=USE_AMP):
            output = active(**batch).logits.float()
        probabilities = torch.softmax(output, dim=-1)
        confidence, prediction = probabilities.max(dim=-1)
        for position, feature in enumerate(chunk):
            length = len(feature["input_ids"])
            logits_by_feature[id(feature)] = (
                prediction[position, :length].cpu().numpy(),
                confidence[position, :length].cpu().numpy(),
            )

    predictions = []
    for index, record in enumerate(rows):
        text = record["source_text"]
        best = {}
        for feature in by_row[index]:
            prediction, confidence = logits_by_feature[id(feature)]
            for (start, end), label_id, score in zip(
                    feature["offset_mapping"], prediction, confidence):
                if start == end:
                    continue
                key = (int(start), int(end))
                if key not in best or float(score) > best[key][1]:
                    best[key] = (int(label_id), float(score))
        offsets = sorted(best)
        if not offsets:
            predictions.append([]); continue
        labels, _ = repair_bio([ID2LABEL[best[o][0]] for o in offsets])
        scores = [best[o][1] for o in offsets]
        consolidated = consolidate_predictions_to_words(text, offsets, labels, scores)
        predictions.append([
            {"label": span.entity_type, "start": span.start_char, "end": span.end_char}
            for span in consolidated.spans
        ])
    return predictions

## 16. Train

One complete run. AdamW with linear warmup over the first 6% of steps then
linear decay - DeBERTa-v3 is warmup-sensitive and destabilises without it. Up to
10 epochs with early stopping after 2 epochs without improvement.

**Resume.** `latest.pt` carries the epoch, optimizer, scheduler, scaler, best
metric, patience state and run signature. A checkpoint that already recorded
early stopping - or that finished the epoch budget - is not resumed into
another epoch; the best checkpoint is loaded and the run continues at
section 17.

**Model selection** uses typed entity F1 on the *combined* validation view
(`val_original` + `val_supplementary`), subsampled per epoch for speed. Selecting
on `val_original` alone would reward IID overfitting; selecting on
`val_supplementary` alone would reward the gap-filler distribution.

In [17]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from privacy_eval.dataset_metrics import evaluate_spans

RUN_EPOCHS = 1 if SMOKE_TEST else MAX_EPOCHS
steps_per_epoch = min(len(train_loader), SMOKE_TRAIN_STEPS) if SMOKE_TEST else len(train_loader)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = max(1, steps_per_epoch * RUN_EPOCHS)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(total_steps * WARMUP_RATIO), total_steps
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP, init_scale=AMP_INITIAL_SCALE)

rng = random.Random(SEED)
epoch_eval_limit = 16 if SMOKE_TEST else EPOCH_EVAL_SAMPLE // 2
epoch_val = (
    rng.sample(val_original, min(epoch_eval_limit, len(val_original)))
    + rng.sample(val_supplementary, min(epoch_eval_limit, len(val_supplementary)))
)
print(f"per-epoch validation sample: {len(epoch_val):,} rows "
      f"(full sets are scored in sections 17-21)")

run_signature = hashlib.sha256(json.dumps({
    "train_sha256": train_artifact["sha256"], "ontology": list(BIO_LABELS),
    "base_model_revision": BASE_MODEL_REVISION, "total_steps": total_steps,
    "batch_size": TRAIN_BATCH_SIZE, "learning_rate": LEARNING_RATE,
    "amp_initial_scale": AMP_INITIAL_SCALE,
}, sort_keys=True).encode("utf-8")).hexdigest()
history, best_metric, best_epoch, patience_left = [], -1.0, -1, EARLY_STOPPING_PATIENCE
BEST_CHECKPOINT = CHECKPOINTS / "best.pt"
LATEST_CHECKPOINT = CHECKPOINTS / "latest.pt"
start_epoch, optimizer_steps = 1, 0

def save_checkpoint(path, payload):
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)

training_complete = False
if RESUME_TRAINING and LATEST_CHECKPOINT.exists():
    resumed = torch.load(LATEST_CHECKPOINT, map_location=device, weights_only=False)
    if resumed.get("run_signature") != run_signature:
        raise RuntimeError("latest.pt belongs to different data/config; move it aside or rebuild intentionally")
    model.load_state_dict(resumed["model_state"]); optimizer.load_state_dict(resumed["optimizer_state"])
    scheduler.load_state_dict(resumed["scheduler_state"]); scaler.load_state_dict(resumed["scaler_state"])
    history = list(resumed["history"]); best_metric = float(resumed["best_metric"])
    best_epoch = int(resumed["best_epoch"]); patience_left = int(resumed["patience_left"])
    start_epoch = int(resumed["epoch"]) + 1
    optimizer_steps = int(resumed.get("optimizer_steps", 0))
    # Early stopping is a decision, not an interruption: a run that already
    # triggered it must not be resumed into another epoch.
    early_stopped = bool(resumed.get("early_stopped", patience_left <= 0))
    if early_stopped:
        training_complete = True
        print(f"latest.pt records early stopping after epoch {resumed['epoch']} "
              f"(patience left {patience_left}); no further epoch will start")
    elif start_epoch > RUN_EPOCHS:
        training_complete = True
        print(f"latest.pt already completed the {RUN_EPOCHS}-epoch budget; nothing left to train")
    else:
        print(f"resuming after epoch {start_epoch - 1}; next epoch is {start_epoch}")

planned_epochs = [] if training_complete else list(range(start_epoch, RUN_EPOCHS + 1))
if training_complete:
    print(f"reusing the finished run: best epoch {best_epoch}, combined typed F1 {best_metric:.4f}")

for epoch in planned_epochs:
    model.train()
    running, steps_seen, started = 0.0, 0, time.perf_counter()
    for step, batch in enumerate(train_loader, start=1):
        batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", enabled=USE_AMP):
            loss = model(**batch).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scale_before = scaler.get_scale()
        scaler.step(optimizer); scaler.update()
        optimizer_step_skipped = USE_AMP and scaler.get_scale() < scale_before
        if not optimizer_step_skipped:
            scheduler.step(); optimizer_steps += 1
        running += loss.item(); steps_seen += 1
        if step % 500 == 0:
            print(f"  epoch {epoch} step {step}/{len(train_loader)} loss {running / step:.4f}")
        if SMOKE_TEST and step >= SMOKE_TRAIN_STEPS:
            break

    metrics = evaluate_spans(epoch_val, predict_spans(epoch_val))
    typed_f1 = metrics["typed_entity"]["f1"]
    entry = {
        "epoch": epoch,
        "train_loss": running / max(1, steps_seen),
        "combined_typed_f1": typed_f1,
        "combined_char_recall": metrics["private_character_recall"],
        "combined_leakage": metrics["private_character_leakage"],
        "hard_negative_fp_rate": metrics["hard_negative_false_positive_rate"],
        "seconds": time.perf_counter() - started,
        "learning_rate": scheduler.get_last_lr()[0],
        "optimizer_steps": optimizer_steps,
    }
    history.append(entry)
    print(f"epoch {epoch}: loss {entry['train_loss']:.4f} | typed F1 {typed_f1:.4f} | "
          f"leakage {entry['combined_leakage']:.4f} | hard-neg FP {entry['hard_negative_fp_rate']:.4f} | "
          f"{entry['seconds']:.0f}s")

    if typed_f1 > best_metric:
        best_metric, best_epoch, patience_left = typed_f1, epoch, EARLY_STOPPING_PATIENCE
        save_checkpoint(BEST_CHECKPOINT, {"epoch": epoch, "model_state": model.state_dict(),
                    "typed_f1": typed_f1, "run_signature": run_signature})
        print(f"  new best; checkpoint saved")
    else:
        patience_left -= 1
        print(f"  no improvement; patience left {patience_left}")
    save_checkpoint(LATEST_CHECKPOINT, {
        "epoch": epoch, "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(), "history": history,
        "best_metric": best_metric, "best_epoch": best_epoch,
        "patience_left": patience_left, "optimizer_steps": optimizer_steps,
        "early_stopped": patience_left <= 0, "max_epochs": RUN_EPOCHS,
        "run_signature": run_signature,
    })
    if patience_left <= 0:
        print(f"early stopping after epoch {epoch}")
        break

if not BEST_CHECKPOINT.exists():
    raise FileNotFoundError(f"No best checkpoint exists at {BEST_CHECKPOINT}")
if SMOKE_TEST:
    assert optimizer_steps > 0, "smoke run never completed an optimizer/scheduler step"
model.load_state_dict(torch.load(BEST_CHECKPOINT, map_location=device, weights_only=False)["model_state"])
model.to(device)
print(f"\nselected epoch {best_epoch} with combined typed F1 {best_metric:.4f}")
(METRICS_DIR / "training_history.json").write_text(
    json.dumps({"history": history, "best_epoch": best_epoch,
                "best_combined_typed_f1": best_metric,
                "selection_metric": "typed entity F1 on combined validation (subsampled)"},
               indent=2) + "\n", encoding="utf-8")

per-epoch validation sample: 6,000 rows (full sets are scored in sections 17-21)


RuntimeError: latest.pt belongs to different data/config; move it aside or rebuild intentionally

### Load Best Checkpoint

If training is manually interrupted after a completed epoch, run this cell before continuing to evaluation/export. It reloads the selected `best.pt` checkpoint so downstream artifacts are never generated from a partially trained in-memory model.

In [35]:
SKIP_TRAINING_USE_EXISTING_BEST = True

if not BEST_CHECKPOINT.exists():
    raise FileNotFoundError(f"No best checkpoint exists at {BEST_CHECKPOINT}")

best_checkpoint_state = torch.load(
    BEST_CHECKPOINT,
    map_location=device,
    weights_only=False
)

if best_checkpoint_state.get("run_signature") != run_signature:
    if SKIP_TRAINING_USE_EXISTING_BEST:
        print("WARNING: best.pt run signature differs from current notebook.")
        print("Proceeding intentionally because existing best checkpoint was requested.")
    else:
        raise RuntimeError(
            "best.pt belongs to different data/config; move it aside or rebuild intentionally"
        )

model.load_state_dict(best_checkpoint_state["model_state"])
model.to(device)
model.eval()

best_epoch = int(best_checkpoint_state["epoch"])
best_metric = float(best_checkpoint_state["typed_f1"])

print(
    f"Loaded best checkpoint: epoch {best_epoch} | "
    f"combined typed F1 {best_metric:.4f}"
)

Proceeding intentionally because existing best checkpoint was requested.
Loaded best checkpoint: epoch 3 | combined typed F1 0.1828


## 17. Evaluate the six sets

Each set is scored separately and kept separately. Sets 1-4 run now and OOD dev
is section 21, all on the selected PyTorch/FP32 checkpoint. The frozen final set
is section 27 and is scored on the accepted INT8 candidate, never here.

In [36]:
from privacy_eval.dataset_metrics import evaluate_spans, password_analysis, restrict_to_labels, subset_by

evaluations = {}
def bounded_eval(rows):
    return rows[:SMOKE_EVAL_ROWS] if SMOKE_TEST else rows
test_eval_rows = bounded_eval(test_rows)
val_orig_eval_rows = bounded_eval(val_original)
val_supp_eval_rows = bounded_eval(val_supplementary)
ood_dev_eval_rows = bounded_eval(ood_dev)


def evaluate_partition(name, rows, *, extras=True):
    started = time.perf_counter()
    predicted = predict_spans(rows)
    metrics = evaluate_spans(rows, predicted)
    metrics["evaluation_seconds"] = time.perf_counter() - started
    if extras:
        metrics["by_source"] = subset_by(rows, predicted, "source_dataset")
        metrics["password"] = password_analysis(rows, predicted)
    evaluations[name] = metrics
    print(f"{name:22} rows {metrics['rows']:>7,} | typed F1 {metrics['typed_entity']['f1']:.4f} "
          f"| char recall {metrics['private_character_recall']:.4f} "
          f"| leakage {metrics['private_character_leakage']:.4f} "
          f"| hard-neg FP {metrics['hard_negative_false_positive_rate']:.4f}")
    return metrics, predicted


print("=== 1. training-side test (AI4Privacy baseline rows + grouped supplementary test rows) ===")
test_metrics, test_predictions = evaluate_partition("test", test_eval_rows)

print("\n=== 2. val_original (40,909 untouched AI4Privacy English validation) ===")
val_orig_metrics, val_orig_predictions = evaluate_partition("val_original", val_orig_eval_rows)

print("\n=== 3. val_supplementary (grouped 20% holdout) ===")
val_supp_metrics, val_supp_predictions = evaluate_partition("val_supplementary", val_supp_eval_rows)

print("\n=== 4. combined Model V2 validation ===")
combined_rows = val_orig_eval_rows + val_supp_eval_rows
combined_predictions = val_orig_predictions + val_supp_predictions
combined_metrics = evaluate_spans(combined_rows, combined_predictions)
combined_metrics["composition"] = {
    "val_original_rows": len(val_orig_eval_rows),
    "val_supplementary_rows": len(val_supp_eval_rows),
    "note": "Reconstructed from the two sets above; never materialised as a third file.",
}
evaluations["combined_validation"] = combined_metrics
print(f"{'combined_validation':22} rows {combined_metrics['rows']:>7,} | "
      f"typed F1 {combined_metrics['typed_entity']['f1']:.4f} | "
      f"char recall {combined_metrics['private_character_recall']:.4f} | "
      f"leakage {combined_metrics['private_character_leakage']:.4f}")

=== 1. training-side test (AI4Privacy baseline rows + grouped supplementary test rows) ===
test                   rows  19,136 | typed F1 0.9549 | char recall 0.9974 | leakage 0.0026 | hard-neg FP 0.2500

=== 2. val_original (40,909 untouched AI4Privacy English validation) ===
val_original           rows  40,909 | typed F1 0.9610 | char recall 0.9983 | leakage 0.0017 | hard-neg FP 0.0000

=== 3. val_supplementary (grouped 20% holdout) ===
val_supplementary      rows   6,907 | typed F1 0.8878 | char recall 0.9858 | leakage 0.0142 | hard-neg FP 0.0000

=== 4. combined Model V2 validation ===
combined_validation    rows  47,816 | typed F1 0.9558 | char recall 0.9971 | leakage 0.0029


## 18. Per-label and error analysis

Per-label numbers are reported against the set that can actually support them:
`val_original` for labels the base distribution teaches, `val_supplementary` for
the weak labels the gap-fillers were selected to strengthen.

In [37]:
print(f"{'label':18} {'val_orig F1':>12} {'supp F1':>10} {'supp sup':>9}  note")
for label in ENTITIES:
    original = val_orig_metrics["per_label"].get(label, {})
    supplementary = val_supp_metrics["per_label"].get(label, {})
    notes = []
    if label in ontology.WEAK_LABELS:
        notes.append("weak")
    if label in ontology.OOD_WEAK_LABELS:
        notes.append(f"ood-weak ({ontology.OOD_WEAK_LABELS[label]:.3f} tinybert)")
    print(f"{label:18} {original.get('f1', float('nan')):>12.4f} "
          f"{supplementary.get('f1', float('nan')):>10.4f} "
          f"{supplementary.get('support', 0):>9,}  {', '.join(notes)}")

print("\nboundary-correct but type-wrong confusions (combined validation, top 12):")
for pair, count in list(combined_metrics["boundary_correct_type_confusions"].items())[:12]:
    print(f"  {pair:36} {count:,}")

print("\nno-PII rows in each evaluation set:")
for name in ("test", "val_original", "val_supplementary"):
    metrics = evaluations[name]
    print(f"  {name:20} {metrics['hard_negative_rows']:>6,} rows | "
          f"FP rate {metrics['hard_negative_false_positive_rate']:.4f}")

label               val_orig F1    supp F1  supp sup  note
ACCOUNTNUM               0.0000     0.9392       566  weak
AGE                      0.9749     0.0000         0  
AMOUNT                   0.0000        nan         0  weak
BANKNAME                 0.0000     0.0000        33  weak
BUILDINGNUM              0.9912     0.0000         0  
CITY                     0.9883     0.7578       519  
COUNTRY                  0.0000     0.8916       827  weak
CREDITCARDNUMBER         0.9611     0.9548       827  
CURRENCY                 0.0000     0.0000         1  weak
DATE                     0.9989     0.9586     5,800  ood-weak (0.275 tinybert)
DRIVERLICENSENUM         0.9050     0.4000         1  ood-weak (0.143 tinybert)
EMAIL                    0.9994     0.9901     1,816  
GENDER                   0.9703     0.7611        50  
GIVENNAME                0.9057     0.3641       300  
IDCARDNUM                0.9083     0.9105     1,009  ood-weak (0.111 tinybert)
IPV4                 

## 19. PASSWORD analysis

`PASSWORD` is the one new label, so it gets its own report. A single F1 would
hide the distinction that matters: catching a disclosed credential is only
useful if talking *about* passwords does not trigger it.

In [38]:
for name in ("test", "val_original", "val_supplementary"):
    analysis = evaluations[name].get("password")
    if not analysis:
        continue
    print(f"--- {name} ---")
    print(f"  credential-disclosure rows      : {analysis['disclosure_rows']:,}")
    if analysis["disclosure_metrics"]:
        disclosure = analysis["disclosure_metrics"]["per_label"].get("PASSWORD", {})
        print(f"  disclosure PASSWORD P/R/F1      : "
              f"{disclosure.get('precision', 0):.4f} / {disclosure.get('recall', 0):.4f} / "
              f"{disclosure.get('f1', 0):.4f}  (support {disclosure.get('support', 0)})")
    print(f"  password-related no-PII rows    : {analysis['password_hard_negative_rows']:,}")
    print(f"  false PASSWORD on those rows    : {analysis['password_hard_negative_false_positives']:,} "
          f"({analysis['password_hard_negative_false_positive_rate']:.4f})")

print("\nA useful PASSWORD label needs high disclosure recall AND a low")
print("hard-negative rate. If the second number is high, the model has learned")
print("the word 'password' rather than the presence of a secret.")

--- test ---
  credential-disclosure rows      : 18
  disclosure PASSWORD P/R/F1      : 1.0000 / 0.9444 / 0.9714  (support 18)
  password-related no-PII rows    : 0
  false PASSWORD on those rows    : 0 (0.0000)
--- val_original ---
  credential-disclosure rows      : 0
  password-related no-PII rows    : 0
  false PASSWORD on those rows    : 0 (0.0000)
--- val_supplementary ---
  credential-disclosure rows      : 174
  disclosure PASSWORD P/R/F1      : 0.9486 / 0.9486 / 0.9486  (support 175)
  password-related no-PII rows    : 0
  false PASSWORD on those rows    : 0 (0.0000)

A useful PASSWORD label needs high disclosure recall AND a low
hard-negative rate. If the second number is high, the model has learned
the word 'password' rather than the presence of a secret.


## 20. TinyBERT comparison

Two restrictions make this comparison honest. The rows are restricted to the
AI4Privacy component of the training-side `test` set, because the supplementary
test rows have no TinyBERT number. The labels are restricted to the 30 shared
entity types, because Model V2 removes `WEIGHT` and adds `PASSWORD`, and one
aggregate figure across two label spaces would be misleading.

A third caveat is new. The section 9 contamination sweep removes any historical
test row that also exists in a protected evaluation set, so the comparison
subset can be slightly smaller than the 16,374 rows the frozen TinyBERT
aggregate was measured on. The cell prints both counts. If they differ, the
comparison is close but **not** exactly like-for-like, and saying so is worth
more than keeping a contaminated row to preserve a round number.

The full Model V2 `test` number, over all 31 types and all test rows, is
reported beside it and is not interchangeable with either.

In [39]:
ai4_pairs = [(row, prediction) for row, prediction in zip(test_eval_rows, test_predictions)
             if row["source_dataset"] == "ai4privacy"]
shared_test = restrict_to_labels([row for row, _ in ai4_pairs], SHARED_WITH_TINYBERT)
shared_predictions = [
    [span for span in prediction if span["label"] in set(SHARED_WITH_TINYBERT)]
    for _, prediction in ai4_pairs
]
shared_metrics = evaluate_spans(shared_test, shared_predictions)
evaluations["test_shared_ontology"] = shared_metrics

print(f"AI4Privacy comparison subset: {len(shared_test):,} of {len(test_eval_rows):,} "
      f"training-side test rows (the rest are supplementary test rows)")
if AI4_TEST_ROWS_REMOVED:
    print(f"  NOT exactly like-for-like: the frozen TinyBERT aggregate used all "
          f"{AI4_TEST_HISTORICAL_ROWS:,} historical rows; {AI4_TEST_ROWS_REMOVED} of those "
          f"also exist in a protected evaluation set and were removed, leaving "
          f"{AI4_TEST_COMPARISON_ROWS:,}.")
else:
    print(f"  Row set is identical to the historical {AI4_TEST_HISTORICAL_ROWS:,} rows.")
print(f"Shared-ontology comparison (30 entity types, {len(shared_test):,} AI4Privacy rows)")
print(f"  Model V2 typed entity F1 : {shared_metrics['typed_entity']['f1']:.4f}")
print(f"  Model V2 char recall     : {shared_metrics['private_character_recall']:.4f}")
print(f"  Model V2 leakage         : {shared_metrics['private_character_leakage']:.4f}")
print("\nFrozen TinyBERT IID baseline (same-source synthetic final test, INT8):")
print("  typed entity F1 0.9496 | char recall 0.9969 | leakage 0.0031")
print("\nFull Model V2 test set (31 types incl. PASSWORD, AI4Privacy + supplementary rows):")
print(f"  typed entity F1 {test_metrics['typed_entity']['f1']:.4f}")
print("\nThese two Model V2 rows are NOT interchangeable. Only the shared-ontology")
print("row may be compared with TinyBERT; the label spaces differ by WEIGHT and PASSWORD.")

AI4Privacy comparison subset: 16,373 of 19,136 training-side test rows (the rest are supplementary test rows)
  NOT exactly like-for-like: the frozen TinyBERT aggregate used all 16,374 historical rows; 1 of those also exist in a protected evaluation set and were removed, leaving 16,373.
Shared-ontology comparison (30 entity types, 16,373 AI4Privacy rows)
  Model V2 typed entity F1 : 0.9613
  Model V2 char recall     : 0.9983
  Model V2 leakage         : 0.0017

Frozen TinyBERT IID baseline (same-source synthetic final test, INT8):
  typed entity F1 0.9496 | char recall 0.9969 | leakage 0.0031

Full Model V2 test set (31 types incl. PASSWORD, AI4Privacy + supplementary rows):
  typed entity F1 0.9549

These two Model V2 rows are NOT interchangeable. Only the shared-ontology
row may be compared with TinyBERT; the label spaces differ by WEIGHT and PASSWORD.


## 21. OOD development set

The generalisation diagnostic. Free to iterate on: error analysis here may guide
data and augmentation design. Reference points from the frozen TinyBERT
baseline, production stack, final partition: typed F1 **0.5665**, character
recall **0.6819**, leakage **0.3181**, hard-negative FP rate **0.6528**.

In [40]:
ood_dev_metrics, ood_dev_predictions = evaluate_partition("ood_dev", ood_dev_eval_rows, extras=False)

by_category = defaultdict(lambda: ([], []))
for record, prediction in zip(ood_dev_eval_rows, ood_dev_predictions):
    bucket = by_category[str(record.get("category", "unknown"))]
    bucket[0].append(record); bucket[1].append(prediction)

print("\nby category:")
for category, (rows, predictions) in sorted(by_category.items()):
    metrics = evaluate_spans(rows, predictions)
    print(f"  {category:26} typed F1 {metrics['typed_entity']['f1']:.4f} | "
          f"char recall {metrics['private_character_recall']:.4f} | "
          f"hard-neg FP {metrics['hard_negative_false_positive_rate']:.4f}")

print("\nTinyBERT production reference (final partition):")
print("  typed F1 0.5665 | char recall 0.6819 | leakage 0.3181 | hard-neg FP 0.6528")
print("Note: these Model V2 numbers are model-only (no structured regex layer),")
print("so the fairer TinyBERT comparison is its model-only INT8 typed F1 of 0.4649.")

ood_dev                rows     360 | typed F1 0.7078 | char recall 0.8335 | leakage 0.1665 | hard-neg FP 0.2222

by category:
  contextual_multi_entity    typed F1 0.9299 | char recall 1.0000 | hard-neg FP 0.0000
  email                      typed F1 0.4371 | char recall 0.8469 | hard-neg FP 0.0000
  hard_negatives             typed F1 0.0000 | char recall 0.0000 | hard-neg FP 0.2222
  long_window                typed F1 1.0000 | char recall 1.0000 | hard-neg FP 0.0000
  names                      typed F1 0.7721 | char recall 0.9951 | hard-neg FP 0.0000
  numeric_structured         typed F1 0.4235 | char recall 0.6223 | hard-neg FP 0.0000

TinyBERT production reference (final partition):
  typed F1 0.5665 | char recall 0.6819 | leakage 0.3181 | hard-neg FP 0.6528
Note: these Model V2 numbers are model-only (no structured regex layer),
so the fairer TinyBERT comparison is its model-only INT8 typed F1 of 0.4649.


## 22. Save model, tokenizer, config and metrics

In [26]:
model.save_pretrained(HF_MODEL_DIR, safe_serialization=True)
tokenizer.save_pretrained(TOKENIZER_DIR)

# Match the preflight's compatibility fix: Transformers 5.x serialises
# extra_special_tokens as a list, which the production-pinned 4.53.3 reads as a
# mapping and rejects. Normalise before the artifact leaves this notebook.
tokenizer_config_path = TOKENIZER_DIR / "tokenizer_config.json"
tokenizer_config = json.loads(tokenizer_config_path.read_text(encoding="utf-8"))
for unstable_key in ("backend", "extra_special_tokens", "is_local", "local_files_only", "vocab_type"):
    tokenizer_config.pop(unstable_key, None)
tokenizer_config_path.write_text(
    json.dumps(tokenizer_config, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
reloaded_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, local_files_only=True, use_fast=True)
assert reloaded_tokenizer.is_fast and len(reloaded_tokenizer) == len(tokenizer)
print("normalised and locally reloaded tokenizer for the production pin")

training_config = {
    "dataset_manifest": MANIFEST_PATH.relative_to(REPO_ROOT).as_posix(),
    "base_model": BASE_MODEL, "base_model_revision": BASE_MODEL_REVISION,
    "num_labels": NUM_LABELS, "entities": list(ENTITIES),
    "seed": SEED, "max_epochs": MAX_EPOCHS, "best_epoch": best_epoch,
    "smoke_test": SMOKE_TEST, "smoke_train_steps": SMOKE_TRAIN_STEPS if SMOKE_TEST else None,
    "run_signature": run_signature, "resume_training": RESUME_TRAINING,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "train_batch_size": TRAIN_BATCH_SIZE, "eval_batch_size": EVAL_BATCH_SIZE,
    "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO, "max_grad_norm": MAX_GRAD_NORM,
    "lr_scheduler": "linear warmup then linear decay",
    "mixed_precision": USE_AMP, "amp_initial_scale": AMP_INITIAL_SCALE,
    "optimizer_steps": optimizer_steps, "context_limit": CONTEXT_LIMIT,
    "overflow_stride": OVERFLOW_STRIDE,
    "train_rows": len(train_rows), "test_rows": len(test_rows),
    "val_original_rows": len(val_original), "val_supplementary_rows": len(val_supplementary),
    "selection_metric": "typed entity F1 on combined validation",
    "best_combined_typed_f1": best_metric,
}
TRAINING_CONFIG_PATH.write_text(json.dumps(training_config, indent=2) + "\n", encoding="utf-8")
(METRICS_DIR / "evaluation_pytorch_fp32.json").write_text(
    json.dumps(evaluations, indent=2, default=float) + "\n", encoding="utf-8"
)
print("saved model, tokenizer, training config and FP32 metrics")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.35s/it]


normalised and locally reloaded tokenizer for the production pin
saved model, tokenizer, training config and FP32 metrics


## 23. Export FP32 ONNX

Modern Dynamo exporter, opset 17, dynamic batch and sequence, output
`logits[batch, sequence, 63]`. `PYTHONUTF8=1` is mandatory here - see section 1.

The export runs on an **independent CPU instance** reloaded from the artifact
saved in section 22. The trained `model` is never moved off its device, so
every later evaluation cell still works after this section has run.

In [27]:
FP32_ONNX = FP32_DIR / "model_fp32.onnx"
INT8_ONNX = INT8_DIR / "model_int8.onnx"

# Export from a separate CPU instance reloaded from the artifact written in
# section 22 (the selected best checkpoint). `model` keeps its device and dtype,
# so sections 17-21 and 26 can be re-run after an export without failing.
export_model = AutoModelForTokenClassification.from_pretrained(
    HF_MODEL_DIR, local_files_only=True
)
export_model = export_model.to(device="cpu", dtype=torch.float32).eval()
assert export_model.config.num_labels == NUM_LABELS
trained_state, export_state = model.state_dict(), export_model.state_dict()
assert set(trained_state) == set(export_state), "the export instance has a different parameter set"
for reference in ("classifier.weight", "classifier.bias"):
    assert torch.equal(trained_state[reference].detach().float().cpu(),
                       export_state[reference].detach().float()), \
        f"{reference} differs between the selected checkpoint and the export instance"
del trained_state, export_state


class LogitsOnly(torch.nn.Module):
    def __init__(self, active_model):
        super().__init__(); self.active_model = active_model
    def forward(self, input_ids, attention_mask):
        return self.active_model(input_ids=input_ids, attention_mask=attention_mask).logits
export_wrapper = LogitsOnly(export_model).eval()
sample = tokenizer(
    ["OnKith Model V2 ONNX export shape check.",
     "A deliberately longer second sentence keeps both exported axes dynamic."],
    return_tensors="pt", padding=True, truncation=True, max_length=CONTEXT_LIMIT,
    return_token_type_ids=False,
)

batch_dim = torch.export.Dim("batch_size", min=1)
sequence_dim = torch.export.Dim("sequence_length", min=1, max=CONTEXT_LIMIT)
export_errors = []
try:
    onnx_program = torch.onnx.export(
        export_wrapper, args=(sample["input_ids"], sample["attention_mask"]),
        input_names=["input_ids", "attention_mask"], output_names=["logits"],
        opset_version=17, dynamo=True, external_data=False,
        dynamic_shapes=({0: batch_dim, 1: sequence_dim}, {0: batch_dim, 1: sequence_dim}),
    )
    onnx_program.save(FP32_ONNX, external_data=False)
    onnx_exporter = "dynamo"
except Exception as exc:
    export_errors.append(f"{type(exc).__name__}: {exc}")
    torch.onnx.export(
        export_wrapper, args=(sample["input_ids"], sample["attention_mask"]), f=str(FP32_ONNX),
        input_names=["input_ids", "attention_mask"], output_names=["logits"],
        opset_version=17, dynamo=False,
        dynamic_axes={"input_ids": {0: "batch_size", 1: "sequence_length"},
                      "attention_mask": {0: "batch_size", 1: "sequence_length"},
                      "logits": {0: "batch_size", 1: "sequence_length"}},
    )
    onnx_exporter = "legacy_dynamic_axes_fallback"

import onnx
graph = onnx.load(str(FP32_ONNX))
onnx.checker.check_model(graph)
print(f"FP32 ONNX: {FP32_ONNX.stat().st_size:,} bytes "
      f"({FP32_ONNX.stat().st_size / 1024**2:.2f} MiB)")
print("inputs :", [i.name for i in graph.graph.input])
print("outputs:", [o.name for o in graph.graph.output])
print("exporter:", onnx_exporter, "| fallback errors:", export_errors)

assert next(export_model.parameters()).device.type == "cpu"
assert next(model.parameters()).device.type == device.type, (
    "the trained model was moved off its device by the export; later evaluation "
    "cells would send tensors to the wrong device"
)
print(f"devices unchanged: trained model on {next(model.parameters()).device}, "
      f"export instance on {next(export_model.parameters()).device}")

Loading weights: 100%|██████████| 200/200 [00:00<00:00, 1503.77it/s]
W0816 16:26:20.931000 21612 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0816 16:26:25.052000 21612 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0816 16:26:25.060000 21612 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0816 16:26:25.066000 21612 Lib\site-packages\torch\onnx\_internal\exporter\_registration

[torch.onnx] Obtain model graph for `LogitsOnly([...]` with `torch.export.export(..., strict=False)`...


W0816 16:26:25.445000 21612 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


[torch.onnx] Obtain model graph for `LogitsOnly([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


C:\Users\yahya\AppData\Local\Programs\Python\Python311\Lib\copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


c:\Users\yahya\VSCodeProjects\OnKith\.venv\Lib\site-packages\torch\onnx\_internal\exporter\_onnx_program.py:487: UserWarning: # The axis name: batch_size will not be used, since it shares the same shape constraints with another axis: batch_size.
  rename_mapping = _dynamic_shapes.create_rename_mapping(
c:\Users\yahya\VSCodeProjects\OnKith\.venv\Lib\site-packages\torch\onnx\_internal\exporter\_onnx_program.py:487: UserWarning: # The axis name: sequence_length will not be used, since it shares the same shape constraints with another axis: sequence_length.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


FP32 ONNX: 282,974,651 bytes (269.87 MiB)
inputs : ['input_ids', 'attention_mask']
outputs: ['logits']
exporter: dynamo | fallback errors: []
devices unchanged: trained model on cuda:0, export instance on cpu


## 24. ONNX parity

The exported graph must agree with PyTorch before anything is quantized. Both
the raw logits and the decoded spans are compared, at the four sequence lengths
the preflight used.

In [28]:
import onnxruntime as ort

session = ort.InferenceSession(str(FP32_ONNX), providers=["CPUExecutionProvider"])
assert [item.name for item in session.get_inputs()] == ["input_ids", "attention_mask"]
assert [item.name for item in session.get_outputs()] == ["logits"]
assert session.get_outputs()[0].type == "tensor(float)"
parity = {"max_abs_logit_delta": {}, "argmax_mismatches": {}}

for batch_size, length in ((1, 75), (1, 221), (2, 221), (1, 289), (1, 512)):
    ids = torch.randint(5, int(export_model.config.vocab_size), (batch_size, length))
    mask = torch.ones_like(ids)
    with torch.no_grad():
        reference = export_model(input_ids=ids, attention_mask=mask).logits.numpy()
    candidate = session.run(["logits"], {"input_ids": ids.numpy(), "attention_mask": mask.numpy()})[0]
    assert tuple(candidate.shape) == (batch_size, length, 63), candidate.shape
    key = f"{batch_size}x{length}"
    parity["max_abs_logit_delta"][key] = float(np.abs(reference - candidate).max())
    parity["argmax_mismatches"][key] = int(
        (reference.argmax(-1) != candidate.argmax(-1)).sum()
    )
    print(f"  {key:>7}: max delta {parity['max_abs_logit_delta'][key]:.3e} | "
          f"argmax mismatches {parity['argmax_mismatches'][key]}")

worst = max(parity["max_abs_logit_delta"].values())
assert worst < 1e-4, f"FP32 ONNX diverges from PyTorch (max delta {worst:.3e})"
print(f"\nparity holds: worst delta {worst:.3e} < 1e-4")
(METRICS_DIR / "onnx_parity.json").write_text(json.dumps(parity, indent=2) + "\n", encoding="utf-8")

     1x75: max delta 2.527e-05 | argmax mismatches 0
    1x221: max delta 2.193e-05 | argmax mismatches 0
    2x221: max delta 3.099e-05 | argmax mismatches 0
    1x289: max delta 2.432e-05 | argmax mismatches 0
    1x512: max delta 2.718e-05 | argmax mismatches 0

parity holds: worst delta 3.099e-05 < 1e-4


325

## 25. INT8 dynamic quantization

Standard ONNX Runtime dynamic quantization, `QInt8`, `per_channel=False`,
`reduce_range=False` - the exact configuration the preflight proved works for
this architecture (49 `MatMulInteger` nodes, roughly 78 MiB).

In [29]:
from onnxruntime.quantization import QuantType, quantize_dynamic

quantize_dynamic(
    model_input=str(FP32_ONNX), model_output=str(INT8_ONNX),
    weight_type=QuantType.QInt8, per_channel=False, reduce_range=False,
)
int8_graph = onnx.load(str(INT8_ONNX))
onnx.checker.check_model(int8_graph)
operators = Counter(node.op_type for node in int8_graph.graph.node)
fp32_bytes, int8_bytes = FP32_ONNX.stat().st_size, INT8_ONNX.stat().st_size
print(f"FP32 {fp32_bytes:,} bytes ({fp32_bytes / 1024**2:.2f} MiB)")
print(f"INT8 {int8_bytes:,} bytes ({int8_bytes / 1024**2:.2f} MiB)  "
      f"compression {fp32_bytes / int8_bytes:.2f}x")
print("MatMulInteger:", operators.get("MatMulInteger", 0),
      "| DynamicQuantizeLinear:", operators.get("DynamicQuantizeLinear", 0))

# Bounded regression for the export/quantization path: the trained model must
# still score through the torch path, on its original device, afterwards.
device_probe = predict_spans(test_eval_rows[:2])
assert next(model.parameters()).device.type == device.type
print(f"post-export torch path still works: {len(device_probe)} rows re-scored on "
      f"{next(model.parameters()).device}")

FP32 282,974,651 bytes (269.87 MiB)
INT8 82,278,782 bytes (78.47 MiB)  compression 3.44x
MatMulInteger: 49 | DynamicQuantizeLinear: 49
post-export torch path still works: 2 rows re-scored on cuda:0


## 26. INT8 re-evaluation and quality gate

Export succeeding is not evidence that INT8 is acceptable. The quantized graph is
re-scored through the production decoding path on every view the model is judged
on - training-side `test`, `val_original`, `val_supplementary`, the combined
validation view used for model selection, and OOD dev - and each is compared with
its FP32 counterpart from sections 17 and 21.

The gate fails if any view loses more than `INT8_MAX_TYPED_F1_DROP` typed entity
F1 or gains more than `INT8_MAX_LEAKAGE_RISE` character leakage. OOD dev is
inside the gate deliberately: INT8 must not be declared acceptable while
out-of-distribution behaviour has collapsed. A failed gate writes no accepted
candidate in section 27, and the separate frozen-benchmark notebook then
refuses to run.

In [41]:
from privacy_eval.model_v2_adapter import ModelV2OnnxMasker

int8_masker = ModelV2OnnxMasker(
    INT8_ONNX, TOKENIZER_DIR, {index: label for index, label in ID2LABEL.items()},
    context_limit=CONTEXT_LIMIT, overflow_stride=OVERFLOW_STRIDE,
)


def onnx_predict(rows, masker):
    predictions = []
    for record in rows:
        result = masker.mask_text(record["source_text"])
        predictions.append([
            {"label": entity["entity_type"], "start": entity["start"], "end": entity["end"]}
            for entity in result.entities
        ])
    return predictions


int8_evaluations, int8_predictions = {}, {}
for name, rows in (("test", test_eval_rows), ("val_original", val_orig_eval_rows),
                   ("val_supplementary", val_supp_eval_rows), ("ood_dev", ood_dev_eval_rows)):
    predicted = onnx_predict(rows, int8_masker)
    int8_predictions[name] = predicted
    int8_evaluations[name] = evaluate_spans(rows, predicted)
    print(f"{name:20} typed F1 {int8_evaluations[name]['typed_entity']['f1']:.4f} | "
          f"leakage {int8_evaluations[name]['private_character_leakage']:.4f}")

# The combined validation view is scored explicitly, exactly as section 17 scores
# it on FP32. The selection metric must never be inferred from an average.
combined_int8_rows = val_orig_eval_rows + val_supp_eval_rows
combined_int8_predictions = int8_predictions["val_original"] + int8_predictions["val_supplementary"]
combined_int8_metrics = evaluate_spans(combined_int8_rows, combined_int8_predictions)
combined_int8_metrics["composition"] = {
    "val_original_rows": len(val_orig_eval_rows),
    "val_supplementary_rows": len(val_supp_eval_rows),
    "note": "Reconstructed from the two sets above; never materialised as a third file.",
}
int8_evaluations["combined_validation"] = combined_int8_metrics
print(f"{'combined_validation':20} typed F1 "
      f"{combined_int8_metrics['typed_entity']['f1']:.4f} | "
      f"leakage {combined_int8_metrics['private_character_leakage']:.4f}")

missing_views = [name for name in INT8_GATE_VIEWS
                 if name not in evaluations or name not in int8_evaluations]
assert not missing_views, f"the INT8 gate cannot be evaluated without {missing_views}"

gate = {}
print(f"\n{'view':20} {'FP32 F1':>8} {'INT8 F1':>8} {'dF1':>8}   "
      f"{'FP32 leak':>9} {'INT8 leak':>9} {'d leak':>8}   gate")
for name in INT8_GATE_VIEWS:
    reference, candidate = evaluations[name], int8_evaluations[name]
    f1_drop = reference["typed_entity"]["f1"] - candidate["typed_entity"]["f1"]
    leak_rise = (candidate["private_character_leakage"]
                 - reference["private_character_leakage"])
    gate[name] = {
        "fp32_typed_f1": reference["typed_entity"]["f1"],
        "int8_typed_f1": candidate["typed_entity"]["f1"],
        "typed_f1_drop": f1_drop,
        "fp32_leakage": reference["private_character_leakage"],
        "int8_leakage": candidate["private_character_leakage"],
        "leakage_rise": leak_rise,
        "max_typed_f1_drop": INT8_MAX_TYPED_F1_DROP,
        "max_leakage_rise": INT8_MAX_LEAKAGE_RISE,
        "passes": f1_drop < INT8_MAX_TYPED_F1_DROP and leak_rise < INT8_MAX_LEAKAGE_RISE,
    }
    print(f"{name:20} {gate[name]['fp32_typed_f1']:>8.4f} {gate[name]['int8_typed_f1']:>8.4f} "
          f"{f1_drop:>+8.4f}   {gate[name]['fp32_leakage']:>9.4f} "
          f"{gate[name]['int8_leakage']:>9.4f} {leak_rise:>+8.4f}   "
          f"{'PASS' if gate[name]['passes'] else 'FAIL'}")

INT8_ACCEPTED = bool(gate) and all(entry["passes"] for entry in gate.values())
print(f"\nINT8 quality gate over {len(gate)} views "
      f"(limits: typed F1 drop < {INT8_MAX_TYPED_F1_DROP}, "
      f"leakage rise < {INT8_MAX_LEAKAGE_RISE}): "
      f"{'PASSED' if INT8_ACCEPTED else 'FAILED'}")
if not INT8_ACCEPTED:
    print("Do not deploy this INT8 graph. Section 27 will not write an accepted")
    print("candidate, so model_v2_frozen_ood_final.ipynb refuses to spend the frozen")
    print("set. Investigate per-label damage, starting with the failing views above.")
(METRICS_DIR / "evaluation_onnx_int8.json").write_text(
    json.dumps({"metrics": int8_evaluations, "gate": gate, "accepted": INT8_ACCEPTED,
                "gate_views": list(INT8_GATE_VIEWS),
                "thresholds": {"max_typed_f1_drop": INT8_MAX_TYPED_F1_DROP,
                               "max_leakage_rise": INT8_MAX_LEAKAGE_RISE}},
               indent=2, default=float) + "\n", encoding="utf-8")

test                 typed F1 0.5877 | leakage 0.0026
val_original         typed F1 0.6028 | leakage 0.0017
val_supplementary    typed F1 0.4007 | leakage 0.0144
ood_dev              typed F1 0.4564 | leakage 0.1268
combined_validation  typed F1 0.5884 | leakage 0.0030

view                  FP32 F1  INT8 F1      dF1   FP32 leak INT8 leak   d leak   gate
test                   0.9549   0.5877  +0.3671      0.0026    0.0026  -0.0000   FAIL
val_original           0.9610   0.6028  +0.3583      0.0017    0.0017  +0.0000   FAIL
val_supplementary      0.8878   0.4007  +0.4870      0.0142    0.0144  +0.0003   FAIL
combined_validation    0.9558   0.5884  +0.3674      0.0029    0.0030  +0.0001   FAIL
ood_dev                0.7078   0.4564  +0.2514      0.1665    0.1268  -0.0397   FAIL

INT8 quality gate over 5 views (limits: typed F1 drop < 0.02, leakage rise < 0.015): FAILED
Do not deploy this INT8 graph. Section 27 will not write an accepted
candidate, so model_v2_frozen_ood_final.ipynb refus

45956

## 27. Accepted INT8 candidate

The last step of candidate construction: record exactly which artifact passed
the section 26 gate, and on what evidence. `accepted_int8_candidate.json` is the
only thing `model_v2_frozen_ood_final.ipynb` trusts. That notebook re-hashes the
INT8 graph, the label map, the tokenizer and the training config against this
file before it will spend the frozen benchmark, so a hand-copied ONNX graph
cannot quietly take the candidate's place.

`accepted: true` is written only when a real run clears the gate. A smoke run
writes the same shape under `artifacts/model_v2/smoke/metrics/` with
`smoke_test: true` and `accepted: false`, so smoke evidence can never be
mistaken for a release candidate.

This notebook stops at the candidate. It never loads or scores the frozen OOD
final annotations, which is why **Run All** is always safe.

In [42]:
from privacy_eval import accepted_candidate

CANDIDATE_MANIFEST_PATH = METRICS_DIR / accepted_candidate.CANDIDATE_MANIFEST_NAME
candidate_manifest = accepted_candidate.build_candidate_manifest(
    repo_root=REPO_ROOT,
    int8_onnx=INT8_ONNX, fp32_onnx=FP32_ONNX,
    tokenizer_dir=TOKENIZER_DIR, label_map_path=LABEL_MAP_PATH,
    training_config_path=TRAINING_CONFIG_PATH, best_checkpoint=BEST_CHECKPOINT,
    ontology_version=ontology.MODEL_V2_VERSION,
    num_entities=len(ENTITIES), num_labels=NUM_LABELS,
    base_model=BASE_MODEL, base_model_revision=BASE_MODEL_REVISION,
    run_signature=run_signature, best_epoch=best_epoch,
    best_combined_typed_f1=best_metric, seed=SEED,
    context_limit=CONTEXT_LIMIT, overflow_stride=OVERFLOW_STRIDE,
    fp32_evaluations=evaluations, int8_evaluations=int8_evaluations,
    gate=gate, gate_views=INT8_GATE_VIEWS,
    max_typed_f1_drop=INT8_MAX_TYPED_F1_DROP,
    max_leakage_rise=INT8_MAX_LEAKAGE_RISE,
    frozen_fixture=FROZEN_FIXTURE_FINGERPRINT,
    int8_accepted=INT8_ACCEPTED, smoke_test=SMOKE_TEST,
    timestamp_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
)
# Belt and braces: a smoke candidate is never acceptable, whatever the gate said.
assert candidate_manifest["accepted"] is (INT8_ACCEPTED and not SMOKE_TEST)
assert not (SMOKE_TEST and candidate_manifest["accepted"])
CANDIDATE_MANIFEST_PATH.write_text(
    json.dumps(candidate_manifest, indent=2, default=float) + "\n", encoding="utf-8")

print("wrote", CANDIDATE_MANIFEST_PATH.relative_to(REPO_ROOT))
print(f"  accepted    : {candidate_manifest['accepted']} | "
      f"smoke_test: {candidate_manifest['smoke_test']}")
print(f"  int8 sha256 : {candidate_manifest['model']['int8_sha256']}")
print(f"  run signature: {candidate_manifest['training']['run_signature'][:16]}...")
print(f"  ontology     : {candidate_manifest['ontology']['version']} | "
      f"{candidate_manifest['ontology']['num_labels']} labels")

if candidate_manifest["accepted"]:
    print("\nThis candidate may now be benchmarked exactly once, by")
    print("model_v2_frozen_ood_final.ipynb. Review every number above first:")
    print("once the frozen set is spent it cannot inform this model generation again.")
else:
    reason = "smoke run" if SMOKE_TEST else "INT8 quality gate failed"
    print(f"\nNOT an accepted release candidate ({reason}).")
    print("model_v2_frozen_ood_final.ipynb will refuse this manifest.")

wrote model\DeBERTa-v3-xsmall\artifacts\model_v2\metrics\accepted_int8_candidate.json
  accepted    : False | smoke_test: False
  int8 sha256 : bc8d1d7a7fb14304e195e47079c3dbb330c627ea4c9753f3af667676048106a8
  run signature: 700f90cd49c04930...
  ontology     : model_v2.ontology.1 | 63 labels

NOT an accepted release candidate (INT8 quality gate failed).
model_v2_frozen_ood_final.ipynb will refuse this manifest.


## 28. Production-pin compatibility

The development environment is not the deployment environment. This confirms the
artifacts load under the pinned production versions before anything is shipped.

In [43]:
continuation = " " + chr(92)   # a trailing backslash, safe inside a raw string
command = [
    r".\model\DeBERTa-v3-xsmall\.venv-production\Scripts\python.exe",
    r"model\DeBERTa-v3-xsmall\deployment\verify_production_runtime.py",
    f"--tokenizer {TOKENIZER_DIR.relative_to(REPO_ROOT)}",
    f"--fp32 {FP32_ONNX.relative_to(REPO_ROOT)}",
    f"--int8 {INT8_ONNX.relative_to(REPO_ROOT)}",
    f"--output {(METRICS_DIR / 'production_runtime_compat.json').relative_to(REPO_ROOT)}",
    f"--expected-label-count {NUM_LABELS}",
]
print("Run from the repository root, in the isolated production-pin environment:")
print()
for position, part in enumerate(command):
    indent = "  " if position == 0 else "    "
    tail = continuation if position < len(command) - 1 else ""
    print(f"{indent}{part}{tail}")
print()
print("Pins: transformers 4.53.3, tokenizers 0.21.4, onnxruntime 1.22.1, sentencepiece 0.2.1")
runtime_python = MODEL_DIR / ".venv-production" / "Scripts" / "python.exe"
if runtime_python.exists():
    import subprocess
    runtime_args = [str(runtime_python), str(MODEL_DIR / "deployment" / "verify_production_runtime.py"),
                    "--tokenizer", str(TOKENIZER_DIR), "--fp32", str(FP32_ONNX),
                    "--int8", str(INT8_ONNX), "--output", str(METRICS_DIR / "production_runtime_compat.json"),
                    "--expected-label-count", str(NUM_LABELS)]
    subprocess.run(runtime_args, cwd=REPO_ROOT, check=True)
    print("production-pin compatibility passed")
else:
    print("Production-pin environment not present; the exact command above remains a required release gate.")
print("This must pass before the artifacts are considered deployable.")

Run from the repository root, in the isolated production-pin environment:

  .\model\DeBERTa-v3-xsmall\.venv-production\Scripts\python.exe \
    model\DeBERTa-v3-xsmall\deployment\verify_production_runtime.py \
    --tokenizer model\DeBERTa-v3-xsmall\artifacts\model_v2\tokenizer \
    --fp32 model\DeBERTa-v3-xsmall\artifacts\model_v2\models\fp32\model_fp32.onnx \
    --int8 model\DeBERTa-v3-xsmall\artifacts\model_v2\models\int8\model_int8.onnx \
    --output model\DeBERTa-v3-xsmall\artifacts\model_v2\metrics\production_runtime_compat.json \
    --expected-label-count 63

Pins: transformers 4.53.3, tokenizers 0.21.4, onnxruntime 1.22.1, sentencepiece 0.2.1
production-pin compatibility passed
This must pass before the artifacts are considered deployable.


## 29. Deployment artifact summary and Raspberry Pi checklist

Development-host measurements are **not** Raspberry Pi measurements. The
preflight measured DeBERTa INT8 at 271 ms for 221 tokens and 849 ms for 512
tokens, single-threaded, on a 14-core Windows host. A 4-core Pi 5 will be
materially slower, so the ~1 s target is plausible around the median and
genuinely uncertain at 512 tokens. Nothing below may be quoted as a Pi result.

In [44]:
def fingerprint(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return {"path": path.relative_to(REPO_ROOT).as_posix(), "bytes": path.stat().st_size,
            "sha256": digest.hexdigest()}

summary = {
    "ontology": {"entities": len(ENTITIES), "labels": NUM_LABELS,
                 "added": list(ontology.ADDED_ENTITIES),
                 "removed": list(ontology.REMOVED_ENTITIES),
                 "absent": list(ontology.ABSENT_ENTITIES)},
    "artifacts": {"fp32_onnx": fingerprint(FP32_ONNX), "int8_onnx": fingerprint(INT8_ONNX),
                  "tokenizer": fingerprint(TOKENIZER_DIR / "tokenizer.json")},
    "int8_accepted": INT8_ACCEPTED,
    "int8_gate": {
        name: {"typed_f1_drop": entry["typed_f1_drop"],
               "leakage_rise": entry["leakage_rise"],
               "passes": entry["passes"]}
        for name, entry in gate.items()
    },
    "accepted_candidate": {
        "manifest": CANDIDATE_MANIFEST_PATH.relative_to(REPO_ROOT).as_posix(),
        "accepted": candidate_manifest["accepted"],
        "smoke_test": candidate_manifest["smoke_test"],
        "int8_sha256": candidate_manifest["model"]["int8_sha256"],
        "run_signature": candidate_manifest["training"]["run_signature"],
    },
    "frozen_ood_final": {
        "status": "not run in this notebook",
        "notebook": "model/DeBERTa-v3-xsmall/model_v2_frozen_ood_final.ipynb",
        "fixture": FROZEN_FIXTURE_FINGERPRINT,
        "policy": "Scored exactly once, on the accepted INT8 candidate, in that notebook.",
    },
    "headline_metrics": {
        name: {"typed_entity_f1": evaluations[name]["typed_entity"]["f1"],
               "private_character_leakage": evaluations[name]["private_character_leakage"],
               "hard_negative_false_positive_rate":
                   evaluations[name]["hard_negative_false_positive_rate"]}
        for name in ("test", "val_original", "val_supplementary", "combined_validation", "ood_dev")
        if name in evaluations
    },
    "production_migration": {
        "status": "NOT PERFORMED - separate reviewed release step",
        "required_changes": [
            "EXPECTED_LABEL_COUNT 67 -> 63 in integrated_pipeline/pii_adapter.py",
            "pii.label_count 67 -> 63 in integrated_pipeline/config.py and config/live_config.json",
            "ship the DeBERTa tokenizer under FullPipeline_V3/models/",
            "swap the INT8 ONNX artifact",
            "re-run FullPipeline_V3/tests/test_pii_regression.py",
        ],
        "note": "The per-label masking policy is already live and is model-independent.",
    },
    "raspberry_pi_checklist": [
        "Copy the INT8 graph and tokenizer to the Pi 5 (4 GB).",
        "Measure median and p95 latency at 75, 221, 289 and 512 tokens, 1 and 4 threads.",
        "Record RSS during inference, and peak RSS with Canary 180M resident.",
        "Run a thermal soak; report sustained rather than burst latency.",
        "Compare against the ~1 s target at the p95 length of 221 tokens.",
        "Only after these numbers exist may any Pi performance claim be made.",
    ],
}
(METRICS_DIR / "deployment_summary.json").write_text(
    json.dumps(summary, indent=2, default=float) + "\n", encoding="utf-8")
print(json.dumps(summary, indent=2, default=float))

{
  "ontology": {
    "entities": 31,
    "labels": 63,
    "added": [
      "PASSWORD"
    ],
    "removed": [
      "ALLERGIES",
      "USERNAME",
      "WEIGHT"
    ],
    "absent": [
      "HEIGHT"
    ]
  },
  "artifacts": {
    "fp32_onnx": {
      "path": "model/DeBERTa-v3-xsmall/artifacts/model_v2/models/fp32/model_fp32.onnx",
      "bytes": 282974651,
      "sha256": "276c5b3b88b555f12977becc3a8d35675684919c7f9e4ee1e9475a062b3d844a"
    },
    "int8_onnx": {
      "path": "model/DeBERTa-v3-xsmall/artifacts/model_v2/models/int8/model_int8.onnx",
      "bytes": 82278782,
      "sha256": "bc8d1d7a7fb14304e195e47079c3dbb330c627ea4c9753f3af667676048106a8"
    },
    "tokenizer": {
      "path": "model/DeBERTa-v3-xsmall/artifacts/model_v2/tokenizer/tokenizer.json",
      "bytes": 8339976,
      "sha256": "8196ef6e837c3ed355ae78bf29a244a33f4dc4179a7c364aedeb0cd6433e9948"
    }
  },
  "int8_accepted": false,
  "int8_gate": {
    "test": {
      "typed_f1_drop": 0.36713993051873495,
  

---

## Done

Committed evidence: `config/model_v2_label_map.json`,
`config/model_v2_data_manifest.json`, `config/model_v2_source_mapping.csv`,
`config/model_v2_training_config.json`, everything in
`artifacts/model_v2/metrics/`, and `artifacts/model_v2/tokenizer/`.

Local and gitignored: `artifacts/model_v2/data/`,
`artifacts/model_v2/checkpoints/`, `artifacts/model_v2/models/huggingface/`,
`artifacts/model_v2/models/fp32/`, `artifacts/model_v2/models/int8/`,
`artifacts/model_v2/cache/`, `artifacts/model_v2/smoke/`, and every corpus under
`model/data/`. A smoke run redirects everything, `config/` included, under
`artifacts/model_v2/smoke/`.

The deployment candidate is `artifacts/model_v2/models/int8/model_int8.onnx`
with `artifacts/model_v2/tokenizer/`; `artifacts/model_v2/models/fp32/model_fp32.onnx`
is the parity reference and `artifacts/model_v2/models/huggingface/` is the
export/recovery source.

Order of record: train, select, FP32 validation, OOD dev, export, parity, INT8,
INT8 gate over all five views, then `metrics/accepted_int8_candidate.json`, which
is written with `accepted: true` only when a real run clears that gate.

The frozen OOD final benchmark is **not** part of this notebook. It lives in
[`model_v2_frozen_ood_final.ipynb`](model_v2_frozen_ood_final.ipynb), verifies the
accepted candidate by hash, and scores the frozen set once. Nothing here reads
those annotations, so **Run All** can never spend the final holdout.

Not done here, on purpose: the production migration from the TinyBERT 67-label
contract to this 63-label model. That is a separate reviewed release.